In [13]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/tushardobal/india-city-air-quality-index-dataset-20152023/india_city_aqi_2015_2023.csv


In [14]:
# PHASE 1: SETUP & DATA CONSOLIDATION

In [15]:
!pip install -q optuna

In [16]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from IPython.display import clear_output
from sklearn.metrics import mean_absolute_error, mean_squared_error, f1_score
from sklearn.multioutput import MultiOutputRegressor
import xgboost as xgb
from statsmodels.tsa.api import VAR
import optuna
import os, warnings, copy
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

os.makedirs('/kaggle/working/figures', exist_ok=True)

Using device: cuda


In [17]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/tushardobal/india-city-air-quality-index-dataset-20152023/india_city_aqi_2015_2023.csv


In [18]:
# Load data

DATA_PATH = '/kaggle/input/datasets/tushardobal/india-city-air-quality-index-dataset-20152023/india_city_aqi_2015_2023.csv'

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

REQUIRED_COLS = ['City', 'Date', 'PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'AQI', 'AQI_Bucket']
missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    print('WARNING - missing expected columns:', missing)
    print('Available columns:', list(df.columns))

(32870, 10)
WARNING - missing expected columns: ['City', 'Date', 'PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'AQI', 'AQI_Bucket']
Available columns: ['city', 'date', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'aqi', 'aqi_category']


In [44]:
# ============================================================
# PHASE 8.1: DATASET FEATURE AUDIT
# CHECK FOR REAL METEOROLOGICAL VARIABLES
# ============================================================

print("=" * 70)
print("DATASET FEATURE AUDIT")
print("=" * 70)

print("\nTOTAL COLUMNS:")
print(len(df.columns))

print("\nALL DATASET COLUMNS:")
for i, col in enumerate(df.columns):
    print(f"{i:02d} -> {col}")

print("\n" + "=" * 70)
print("METEOROLOGICAL VARIABLE SEARCH")
print("=" * 70)

meteorological_keywords = [
    'temp',
    'temperature',
    'humidity',
    'humid',
    'wind',
    'windspeed',
    'wind_speed',
    'pressure',
    'rain',
    'rainfall',
    'precipitation',
    'precip'
]

meteorological_columns = []

for col in df.columns:
    col_lower = str(col).lower()

    if any(
        keyword in col_lower
        for keyword in meteorological_keywords
    ):
        meteorological_columns.append(col)

if len(meteorological_columns) > 0:

    print("\nMETEOROLOGICAL COLUMNS FOUND:")

    for col in meteorological_columns:
        print("FOUND ->", col)

else:

    print("\nNO METEOROLOGICAL COLUMNS FOUND")


print("\n" + "=" * 70)

print("CURRENT FEATURE_COLUMNS:")

try:
    print(FEATURE_COLUMNS)

    print(
        "\nNUMBER OF CURRENT FEATURES:",
        len(FEATURE_COLUMNS)
    )

except NameError:

    print(
        "FEATURE_COLUMNS is not defined at this notebook stage."
    )


print("\n" + "=" * 70)

print("CURRENT POLLUTANTS:")

try:
    print(POLLUTANTS)

    print(
        "\nNUMBER OF POLLUTANTS:",
        len(POLLUTANTS)
    )

except NameError:

    print(
        "POLLUTANTS is not defined at this notebook stage."
    )

print("=" * 70)

DATASET FEATURE AUDIT

TOTAL COLUMNS:
11

ALL DATASET COLUMNS:
00 -> City
01 -> Date
02 -> PM2.5
03 -> PM10
04 -> NO2
05 -> SO2
06 -> CO
07 -> O3
08 -> AQI
09 -> AQI_Bucket
10 -> AQI_Class

METEOROLOGICAL VARIABLE SEARCH

NO METEOROLOGICAL COLUMNS FOUND

CURRENT FEATURE_COLUMNS:
FEATURE_COLUMNS is not defined at this notebook stage.

CURRENT POLLUTANTS:
['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3']

NUMBER OF POLLUTANTS: 6


In [45]:
# ============================================================
# PHASE 8B.1: CITY + DATE STRUCTURE AUDIT
# REQUIRED BEFORE METEOROLOGICAL MERGE
# ============================================================

print("=" * 70)
print("CITY + DATE STRUCTURE AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Detect active dataframe
# ------------------------------------------------------------

dataframe_candidates = {
    name: obj
    for name, obj in globals().items()
    if isinstance(obj, pd.DataFrame)
}

print("\nAVAILABLE DATAFRAMES:")

for name, obj in dataframe_candidates.items():
    print(
        f"{name:25s} -> "
        f"shape={obj.shape}"
    )


# ------------------------------------------------------------
# 2. Use df if available
# ------------------------------------------------------------

if 'df' in dataframe_candidates:

    audit_df = dataframe_candidates['df']
    audit_df_name = 'df'

else:

    # Choose largest dataframe
    audit_df_name = max(
        dataframe_candidates,
        key=lambda name: len(dataframe_candidates[name])
    )

    audit_df = dataframe_candidates[audit_df_name]


print("\nSELECTED DATAFRAME:")
print(audit_df_name)

print("\nSHAPE:")
print(audit_df.shape)


# ------------------------------------------------------------
# 3. Print columns
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("COLUMNS")
print("=" * 70)

for i, col in enumerate(audit_df.columns):

    print(
        f"{i:02d} -> {col}"
    )


# ------------------------------------------------------------
# 4. Detect city columns
# ------------------------------------------------------------

city_keywords = [
    'city',
    'station',
    'location',
    'place'
]

city_columns = [
    col
    for col in audit_df.columns
    if any(
        keyword in str(col).lower()
        for keyword in city_keywords
    )
]


print("\n" + "=" * 70)
print("POSSIBLE CITY COLUMNS")
print("=" * 70)

if len(city_columns) > 0:

    for col in city_columns:

        print("\nCOLUMN:", col)

        print(
            "UNIQUE VALUES:",
            audit_df[col].nunique()
        )

        print(
            "SAMPLE VALUES:"
        )

        print(
            audit_df[col]
            .dropna()
            .astype(str)
            .unique()[:20]
        )

else:

    print(
        "NO CITY/STATION COLUMN DETECTED"
    )


# ------------------------------------------------------------
# 5. Detect date columns
# ------------------------------------------------------------

date_keywords = [
    'date',
    'time',
    'year',
    'month',
    'day'
]

date_columns = [
    col
    for col in audit_df.columns
    if any(
        keyword in str(col).lower()
        for keyword in date_keywords
    )
]


print("\n" + "=" * 70)
print("POSSIBLE DATE/TIME COLUMNS")
print("=" * 70)

if len(date_columns) > 0:

    for col in date_columns:

        print("\nCOLUMN:", col)

        print(
            "DTYPE:",
            audit_df[col].dtype
        )

        print(
            "FIRST 5 VALUES:"
        )

        print(
            audit_df[col].head()
        )

else:

    print(
        "NO DATE/TIME COLUMN DETECTED"
    )


# ------------------------------------------------------------
# 6. Dataset date range
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATE RANGE CHECK")
print("=" * 70)

for col in date_columns:

    converted_dates = pd.to_datetime(
        audit_df[col],
        errors='coerce'
    )

    valid_dates = converted_dates.dropna()

    if len(valid_dates) > 0:

        print("\nPOSSIBLE DATE COLUMN:", col)

        print(
            "START DATE:",
            valid_dates.min()
        )

        print(
            "END DATE:",
            valid_dates.max()
        )

        print(
            "VALID DATE ROWS:",
            len(valid_dates)
        )


print("\n" + "=" * 70)
print("AUDIT COMPLETE")
print("=" * 70)

CITY + DATE STRUCTURE AUDIT

AVAILABLE DATAFRAMES:
df                        -> shape=(32870, 11)
pivot                     -> shape=(3287, 80)
var_input                 -> shape=(2300, 2)
cv_df                     -> shape=(5, 5)

SELECTED DATAFRAME:
df

SHAPE:
(32870, 11)

COLUMNS
00 -> City
01 -> Date
02 -> PM2.5
03 -> PM10
04 -> NO2
05 -> SO2
06 -> CO
07 -> O3
08 -> AQI
09 -> AQI_Bucket
10 -> AQI_Class

POSSIBLE CITY COLUMNS

COLUMN: City
UNIQUE VALUES: 10
SAMPLE VALUES:
['Delhi' 'Mumbai' 'Bengaluru' 'Kolkata' 'Chennai' 'Hyderabad' 'Lucknow'
 'Jaipur' 'Pune' 'Ahmedabad']

POSSIBLE DATE/TIME COLUMNS

COLUMN: Date
DTYPE: datetime64[ns]
FIRST 5 VALUES:
0   2015-01-01
1   2015-01-02
2   2015-01-03
3   2015-01-04
4   2015-01-05
Name: Date, dtype: datetime64[ns]

DATE RANGE CHECK

POSSIBLE DATE COLUMN: Date
START DATE: 2015-01-01 00:00:00
END DATE: 2023-12-31 00:00:00
VALID DATE ROWS: 32870

AUDIT COMPLETE


In [46]:
# ============================================================
# PHASE 8B.2: DOWNLOAD REAL HISTORICAL METEOROLOGICAL DATA
# CITY-DATE MATCHED WEATHER FEATURES
# ============================================================

import requests
import pandas as pd
import numpy as np
import time

print("=" * 70)
print("DOWNLOADING HISTORICAL METEOROLOGICAL DATA")
print("=" * 70)

# ------------------------------------------------------------
# CITY COORDINATES
# ------------------------------------------------------------

CITY_COORDINATES = {
    'Delhi': (28.6139, 77.2090),
    'Mumbai': (19.0760, 72.8777),
    'Bengaluru': (12.9716, 77.5946),
    'Kolkata': (22.5726, 88.3639),
    'Chennai': (13.0827, 80.2707),
    'Hyderabad': (17.3850, 78.4867),
    'Lucknow': (26.8467, 80.9462),
    'Jaipur': (26.9124, 75.7873),
    'Pune': (18.5204, 73.8567),
    'Ahmedabad': (23.0225, 72.5714)
}


# ------------------------------------------------------------
# DATE RANGE
# ------------------------------------------------------------

START_DATE = "2015-01-01"
END_DATE = "2023-12-31"


# ------------------------------------------------------------
# OPEN-METEO HISTORICAL API
# ------------------------------------------------------------

WEATHER_URL = (
    "https://archive-api.open-meteo.com/v1/archive"
)


weather_frames = []


# ------------------------------------------------------------
# DOWNLOAD CITY-BY-CITY
# ------------------------------------------------------------

for city, coordinates in CITY_COORDINATES.items():

    latitude, longitude = coordinates

    print()
    print("-" * 70)
    print("DOWNLOADING:", city)
    print("LATITUDE:", latitude)
    print("LONGITUDE:", longitude)
    print("-" * 70)


    params = {

        "latitude": latitude,

        "longitude": longitude,

        "start_date": START_DATE,

        "end_date": END_DATE,

        "daily": [
            "temperature_2m_mean",
            "relative_humidity_2m_mean",
            "wind_speed_10m_mean",
            "surface_pressure_mean",
            "precipitation_sum"
        ],

        "timezone": "Asia/Kolkata"
    }


    try:

        response = requests.get(
            WEATHER_URL,
            params=params,
            timeout=120
        )


        print(
            "HTTP STATUS:",
            response.status_code
        )


        response.raise_for_status()


        weather_json = response.json()


        if "daily" not in weather_json:

            print(
                "WARNING: No daily data returned for",
                city
            )

            continue


        daily_data = weather_json["daily"]


        city_weather = pd.DataFrame(
            daily_data
        )


        city_weather["City"] = city


        city_weather = city_weather.rename(
            columns={

                "time":
                    "Date",

                "temperature_2m_mean":
                    "Temperature",

                "relative_humidity_2m_mean":
                    "Humidity",

                "wind_speed_10m_mean":
                    "WindSpeed",

                "surface_pressure_mean":
                    "Pressure",

                "precipitation_sum":
                    "Rainfall"
            }
        )


        city_weather["Date"] = pd.to_datetime(
            city_weather["Date"]
        )


        city_weather = city_weather[
            [
                "City",
                "Date",
                "Temperature",
                "Humidity",
                "WindSpeed",
                "Pressure",
                "Rainfall"
            ]
        ]


        weather_frames.append(
            city_weather
        )


        print(
            "ROWS DOWNLOADED:",
            len(city_weather)
        )


        print(
            "DATE RANGE:",
            city_weather["Date"].min(),
            "to",
            city_weather["Date"].max()
        )


        print(
            "SUCCESS:",
            city
        )


        time.sleep(1)


    except Exception as e:

        print(
            "ERROR FOR",
            city,
            ":",
            str(e)
        )


# ------------------------------------------------------------
# COMBINE ALL CITIES
# ------------------------------------------------------------

if len(weather_frames) == 0:

    raise RuntimeError(
        "NO WEATHER DATA WAS DOWNLOADED. "
        "CHECK KAGGLE INTERNET ACCESS OR API RESPONSE."
    )


weather_df = pd.concat(
    weather_frames,
    ignore_index=True
)


# ------------------------------------------------------------
# SORT
# ------------------------------------------------------------

weather_df = weather_df.sort_values(
    [
        "City",
        "Date"
    ]
).reset_index(
    drop=True
)


# ------------------------------------------------------------
# SAVE BACKUP
# ------------------------------------------------------------

weather_df.to_csv(
    "/kaggle/working/india_10_city_weather_2015_2023.csv",
    index=False
)


# ------------------------------------------------------------
# FINAL AUDIT
# ------------------------------------------------------------

print()
print("=" * 70)
print("WEATHER DOWNLOAD COMPLETE")
print("=" * 70)


print(
    "WEATHER SHAPE:",
    weather_df.shape
)


print(
    "\nWEATHER COLUMNS:"
)

print(
    weather_df.columns.tolist()
)


print(
    "\nCITIES:"
)

print(
    weather_df["City"].unique()
)


print(
    "\nNUMBER OF CITIES:",
    weather_df["City"].nunique()
)


print(
    "\nDATE RANGE:"
)

print(
    weather_df["Date"].min(),
    "TO",
    weather_df["Date"].max()
)


print(
    "\nMISSING VALUES:"
)

print(
    weather_df[
        [
            "Temperature",
            "Humidity",
            "WindSpeed",
            "Pressure",
            "Rainfall"
        ]
    ].isna().sum()
)


print(
    "\nFIRST 10 ROWS:"
)

display(
    weather_df.head(10)
)


print()
print("=" * 70)
print("PHASE 8B.2 COMPLETE")
print("=" * 70)

DOWNLOADING HISTORICAL METEOROLOGICAL DATA

----------------------------------------------------------------------
DOWNLOADING: Delhi
LATITUDE: 28.6139
LONGITUDE: 77.209
----------------------------------------------------------------------
HTTP STATUS: 200
ROWS DOWNLOADED: 3287
DATE RANGE: 2015-01-01 00:00:00 to 2023-12-31 00:00:00
SUCCESS: Delhi

----------------------------------------------------------------------
DOWNLOADING: Mumbai
LATITUDE: 19.076
LONGITUDE: 72.8777
----------------------------------------------------------------------
HTTP STATUS: 200
ROWS DOWNLOADED: 3287
DATE RANGE: 2015-01-01 00:00:00 to 2023-12-31 00:00:00
SUCCESS: Mumbai

----------------------------------------------------------------------
DOWNLOADING: Bengaluru
LATITUDE: 12.9716
LONGITUDE: 77.5946
----------------------------------------------------------------------
HTTP STATUS: 200
ROWS DOWNLOADED: 3287
DATE RANGE: 2015-01-01 00:00:00 to 2023-12-31 00:00:00
SUCCESS: Bengaluru

------------------------

,City,Date,Temperature,Humidity,WindSpeed,Pressure,Rainfall
0,Bengaluru,2015-01-01,21.7,79,9.9,910.3,4.8
1,Bengaluru,2015-01-02,21.8,75,7.6,912.5,0.8
2,Bengaluru,2015-01-03,21.5,71,8.4,913.6,0.0
3,Bengaluru,2015-01-04,21.3,70,8.1,913.4,0.0
4,Bengaluru,2015-01-05,21.8,69,9.7,913.5,0.0
5,Bengaluru,2015-01-06,21.2,68,7.2,913.7,0.0
6,Bengaluru,2015-01-07,21.3,64,5.6,913.6,0.0
7,Bengaluru,2015-01-08,21.1,68,6.7,913.5,0.0
8,Bengaluru,2015-01-09,20.9,67,7.4,914.6,0.0
9,Bengaluru,2015-01-10,20.5,59,8.9,914.7,0.0



PHASE 8B.2 COMPLETE


In [47]:
# ============================================================
# PHASE 8B.2 FIX:
# RETRY ONLY THE 4 MISSING WEATHER CITIES
# ============================================================

import requests
import pandas as pd
import time

print("=" * 70)
print("RETRYING MISSING WEATHER CITIES")
print("=" * 70)

# ------------------------------------------------------------
# ONLY THE 4 MISSING CITIES
# ------------------------------------------------------------

MISSING_CITY_COORDINATES = {
    'Ahmedabad': (23.0225, 72.5714),
    'Jaipur': (26.9124, 75.7873),
    'Lucknow': (26.8467, 80.9462),
    'Pune': (18.5204, 73.8567)
}

START_DATE = "2015-01-01"
END_DATE = "2023-12-31"

WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"

retry_weather_frames = []

# ------------------------------------------------------------
# DOWNLOAD MISSING CITIES ONE BY ONE
# ------------------------------------------------------------

for city, coordinates in MISSING_CITY_COORDINATES.items():

    latitude, longitude = coordinates

    print()
    print("-" * 70)
    print("RETRYING:", city)
    print("LATITUDE:", latitude)
    print("LONGITUDE:", longitude)
    print("-" * 70)

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": [
            "temperature_2m_mean",
            "relative_humidity_2m_mean",
            "wind_speed_10m_mean",
            "surface_pressure_mean",
            "precipitation_sum"
        ],
        "timezone": "Asia/Kolkata"
    }

    success = False

    for attempt in range(1, 6):

        print(f"Attempt {attempt}/5")

        try:

            response = requests.get(
                WEATHER_URL,
                params=params,
                timeout=180
            )

            print("HTTP STATUS:", response.status_code)

            if response.status_code == 429:
                print("API rate limit detected.")
                print("Waiting 30 seconds...")
                time.sleep(30)
                continue

            response.raise_for_status()

            weather_json = response.json()

            if "daily" not in weather_json:
                print("No daily weather data returned.")
                time.sleep(15)
                continue

            city_weather = pd.DataFrame(
                weather_json["daily"]
            )

            city_weather["City"] = city

            city_weather = city_weather.rename(
                columns={
                    "time": "Date",
                    "temperature_2m_mean": "Temperature",
                    "relative_humidity_2m_mean": "Humidity",
                    "wind_speed_10m_mean": "WindSpeed",
                    "surface_pressure_mean": "Pressure",
                    "precipitation_sum": "Rainfall"
                }
            )

            city_weather["Date"] = pd.to_datetime(
                city_weather["Date"]
            )

            city_weather = city_weather[
                [
                    "City",
                    "Date",
                    "Temperature",
                    "Humidity",
                    "WindSpeed",
                    "Pressure",
                    "Rainfall"
                ]
            ]

            retry_weather_frames.append(
                city_weather
            )

            print("ROWS DOWNLOADED:", len(city_weather))

            print(
                "DATE RANGE:",
                city_weather["Date"].min(),
                "TO",
                city_weather["Date"].max()
            )

            print("SUCCESS:", city)

            success = True

            break

        except Exception as e:

            print(
                "ATTEMPT FAILED:",
                str(e)
            )

            if attempt < 5:
                print("Waiting 20 seconds before retry...")
                time.sleep(20)

    if not success:
        print("FAILED AFTER 5 ATTEMPTS:", city)

    print("Waiting before next city...")
    time.sleep(15)


# ------------------------------------------------------------
# COMBINE RETRIED CITIES
# ------------------------------------------------------------

if len(retry_weather_frames) > 0:

    retry_weather_df = pd.concat(
        retry_weather_frames,
        ignore_index=True
    )

    print()
    print("RETRIED WEATHER SHAPE:")
    print(retry_weather_df.shape)

else:

    raise RuntimeError(
        "NONE OF THE MISSING CITIES COULD BE DOWNLOADED."
    )


# ------------------------------------------------------------
# ADD TO EXISTING WEATHER DATA
# ------------------------------------------------------------

weather_df = pd.concat(
    [
        weather_df,
        retry_weather_df
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# REMOVE ACCIDENTAL DUPLICATES
# ------------------------------------------------------------

weather_df = weather_df.drop_duplicates(
    subset=[
        "City",
        "Date"
    ],
    keep="last"
)


# ------------------------------------------------------------
# SORT
# ------------------------------------------------------------

weather_df = weather_df.sort_values(
    [
        "City",
        "Date"
    ]
).reset_index(
    drop=True
)


# ------------------------------------------------------------
# SAVE COMPLETE WEATHER DATA
# ------------------------------------------------------------

weather_df.to_csv(
    "/kaggle/working/india_10_city_weather_2015_2023.csv",
    index=False
)


# ------------------------------------------------------------
# FINAL STRICT CHECK
# ------------------------------------------------------------

expected_cities = set(
    [
        "Delhi",
        "Mumbai",
        "Bengaluru",
        "Kolkata",
        "Chennai",
        "Hyderabad",
        "Lucknow",
        "Jaipur",
        "Pune",
        "Ahmedabad"
    ]
)

downloaded_cities = set(
    weather_df["City"].unique()
)

still_missing = (
    expected_cities -
    downloaded_cities
)


print()
print("=" * 70)
print("FINAL WEATHER DATA AUDIT")
print("=" * 70)

print(
    "\nWEATHER SHAPE:",
    weather_df.shape
)

print(
    "\nCITIES:"
)

print(
    weather_df["City"].unique()
)

print(
    "\nNUMBER OF CITIES:",
    weather_df["City"].nunique()
)

print(
    "\nSTILL MISSING:"
)

print(
    still_missing
)

print(
    "\nROWS PER CITY:"
)

print(
    weather_df.groupby("City").size()
)

print(
    "\nDUPLICATE CITY-DATE ROWS:"
)

print(
    weather_df.duplicated(
        subset=[
            "City",
            "Date"
        ]
    ).sum()
)

print(
    "\nMISSING WEATHER VALUES:"
)

print(
    weather_df[
        [
            "Temperature",
            "Humidity",
            "WindSpeed",
            "Pressure",
            "Rainfall"
        ]
    ].isna().sum()
)


# ------------------------------------------------------------
# PASS / FAIL
# ------------------------------------------------------------

if len(still_missing) == 0:

    print()
    print("=" * 70)
    print("SUCCESS: ALL 10 CITIES ARE AVAILABLE")
    print("WEATHER DATA IS READY FOR CITY-DATE MERGE")
    print("=" * 70)

else:

    print()
    print("=" * 70)
    print("STOP: SOME CITIES ARE STILL MISSING")
    print("DO NOT MERGE WEATHER DATA YET")
    print("=" * 70)

RETRYING MISSING WEATHER CITIES

----------------------------------------------------------------------
RETRYING: Ahmedabad
LATITUDE: 23.0225
LONGITUDE: 72.5714
----------------------------------------------------------------------
Attempt 1/5
HTTP STATUS: 200
ROWS DOWNLOADED: 3287
DATE RANGE: 2015-01-01 00:00:00 TO 2023-12-31 00:00:00
SUCCESS: Ahmedabad
Waiting before next city...

----------------------------------------------------------------------
RETRYING: Jaipur
LATITUDE: 26.9124
LONGITUDE: 75.7873
----------------------------------------------------------------------
Attempt 1/5
HTTP STATUS: 200
ROWS DOWNLOADED: 3287
DATE RANGE: 2015-01-01 00:00:00 TO 2023-12-31 00:00:00
SUCCESS: Jaipur
Waiting before next city...

----------------------------------------------------------------------
RETRYING: Lucknow
LATITUDE: 26.8467
LONGITUDE: 80.9462
----------------------------------------------------------------------
Attempt 1/5
HTTP STATUS: 200
ROWS DOWNLOADED: 3287
DATE RANGE: 2015-01

In [48]:
# ============================================================
# PHASE 8B.3: MERGE REAL WEATHER DATA + REBUILD MODEL INPUT
# 6 POLLUTANTS + 5 WEATHER + 10 TEMPORAL = 21 CHANNELS
# ============================================================

print("=" * 70)
print("MERGING AQI + METEOROLOGICAL DATA")
print("=" * 70)

# ============================================================
# STEP 1: SAFE CITY-DATE MERGE
# ============================================================

df_weather = df.merge(
    weather_df,
    on=["City", "Date"],
    how="left",
    validate="one_to_one"
)

print("Original AQI shape:", df.shape)
print("Merged shape:", df_weather.shape)

assert len(df_weather) == len(df), (
    f"ROW COUNT CHANGED: df={len(df)}, "
    f"df_weather={len(df_weather)}"
)

WEATHER_COLUMNS = [
    "Temperature",
    "Humidity",
    "WindSpeed",
    "Pressure",
    "Rainfall"
]

print("\nMissing weather values after merge:")
print(df_weather[WEATHER_COLUMNS].isna().sum())

assert df_weather[WEATHER_COLUMNS].isna().sum().sum() == 0, (
    "MISSING WEATHER VALUES FOUND AFTER MERGE"
)

print("\nCITY-DATE WEATHER MERGE PASSED")


# ============================================================
# STEP 2: SORT EXACTLY BY DATE + CITY
# ============================================================

df_weather = df_weather.sort_values(
    ["Date", "City"]
).reset_index(drop=True)


# ============================================================
# STEP 3: REBUILD PIVOT
# ============================================================

MODEL_FEATURES = POLLUTANTS + WEATHER_COLUMNS

print("\nMODEL FEATURES:")
print(MODEL_FEATURES)

print("\nBase feature count:", len(MODEL_FEATURES))

pivot_weather = df_weather.pivot(
    index="Date",
    columns="City",
    values=MODEL_FEATURES + ["AQI", "AQI_Class"]
)

pivot_weather = pivot_weather.sort_index()

cities_weather = sorted(
    df_weather["City"].unique()
)

print("\nCities:", cities_weather)
print("Number of cities:", len(cities_weather))
print("Pivot shape:", pivot_weather.shape)


# ============================================================
# STEP 4: BUILD POLLUTANT + WEATHER TENSOR
# ============================================================

T_total_weather = len(pivot_weather)
M_weather = len(cities_weather)

X_base_weather = np.zeros(
    (
        T_total_weather,
        M_weather,
        len(MODEL_FEATURES)
    ),
    dtype=np.float32
)

for fi, feature in enumerate(MODEL_FEATURES):

    X_base_weather[:, :, fi] = (
        pivot_weather[feature][cities_weather].values
    )


# ============================================================
# STEP 5: REBUILD TEMPORAL FEATURES
# SAME 10 TEMPORAL CHANNELS USED IN CURRENT V2 PIPELINE
# ============================================================

dates_weather = pd.DatetimeIndex(
    pivot_weather.index
)

month = dates_weather.month.values
dow = dates_weather.dayofweek.values
doy = dates_weather.dayofyear.values

month_sin = np.sin(
    2 * np.pi * month / 12
)

month_cos = np.cos(
    2 * np.pi * month / 12
)

dow_sin = np.sin(
    2 * np.pi * dow / 7
)

dow_cos = np.cos(
    2 * np.pi * dow / 7
)

doy_sin = np.sin(
    2 * np.pi * doy / 365.25
)

doy_cos = np.cos(
    2 * np.pi * doy / 365.25
)

season = (
    (month % 12) // 3
).astype(int)

season_onehot = np.eye(
    4,
    dtype=np.float32
)[season]

temporal_base_weather = np.column_stack(
    [
        month_sin,
        month_cos,
        dow_sin,
        dow_cos,
        doy_sin,
        doy_cos,
        season_onehot
    ]
).astype(np.float32)

print(
    "\nTemporal base shape:",
    temporal_base_weather.shape
)

assert temporal_base_weather.shape[1] == 10, (
    f"Expected 10 temporal channels, "
    f"got {temporal_base_weather.shape[1]}"
)

temporal_features_weather = np.repeat(
    temporal_base_weather[:, None, :],
    M_weather,
    axis=1
)


# ============================================================
# STEP 6: FINAL 21-CHANNEL INPUT
# ============================================================

X_weather = np.concatenate(
    [
        X_base_weather,
        temporal_features_weather
    ],
    axis=2
).astype(np.float32)

P_weather = X_weather.shape[2]

print("\n" + "=" * 70)
print("NEW INPUT CHANNEL AUDIT")
print("=" * 70)

print("Pollutant channels:", len(POLLUTANTS))
print("Weather channels:", len(WEATHER_COLUMNS))
print("Temporal channels:", temporal_features_weather.shape[2])
print("TOTAL CHANNELS:", P_weather)
print("X_weather shape:", X_weather.shape)

assert P_weather == 21, (
    f"EXPECTED 21 CHANNELS BUT GOT {P_weather}"
)


# ============================================================
# STEP 7: REBUILD AQI TARGETS
# ============================================================

Y_reg_weather = np.zeros(
    (
        T_total_weather,
        M_weather
    ),
    dtype=np.float32
)

Y_cls_weather = np.zeros(
    (
        T_total_weather,
        M_weather
    ),
    dtype=np.int64
)

Y_reg_weather[:, :] = (
    pivot_weather["AQI"][cities_weather].values
)

Y_cls_weather[:, :] = (
    pivot_weather["AQI_Class"][cities_weather].values
)


# ============================================================
# STEP 8: STRICT FINAL CHECK
# ============================================================

assert X_weather.shape[0] == Y_reg_weather.shape[0]
assert X_weather.shape[0] == Y_cls_weather.shape[0]

assert X_weather.shape[1] == Y_reg_weather.shape[1]
assert X_weather.shape[1] == Y_cls_weather.shape[1]

print()
print("=" * 70)
print("PHASE 8B.3 SUCCESS")
print("=" * 70)

print("X_weather:", X_weather.shape)
print("Y_reg_weather:", Y_reg_weather.shape)
print("Y_cls_weather:", Y_cls_weather.shape)

print()
print("FEATURE PIPELINE:")
print("6 POLLUTANTS + 5 WEATHER + 10 TEMPORAL = 21 CHANNELS")

print()
print("WEATHER-INTEGRATED INPUT IS READY")
print("=" * 70)

MERGING AQI + METEOROLOGICAL DATA
Original AQI shape: (32870, 11)
Merged shape: (32870, 16)

Missing weather values after merge:
Temperature    0
Humidity       0
WindSpeed      0
Pressure       0
Rainfall       0
dtype: int64

CITY-DATE WEATHER MERGE PASSED

MODEL FEATURES:
['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'Temperature', 'Humidity', 'WindSpeed', 'Pressure', 'Rainfall']

Base feature count: 11

Cities: ['Ahmedabad', 'Bengaluru', 'Chennai', 'Delhi', 'Hyderabad', 'Jaipur', 'Kolkata', 'Lucknow', 'Mumbai', 'Pune']
Number of cities: 10
Pivot shape: (3287, 130)

Temporal base shape: (3287, 10)

NEW INPUT CHANNEL AUDIT
Pollutant channels: 6
Weather channels: 5
Temporal channels: 10
TOTAL CHANNELS: 21
X_weather shape: (3287, 10, 21)

PHASE 8B.3 SUCCESS
X_weather: (3287, 10, 21)
Y_reg_weather: (3287, 10)
Y_cls_weather: (3287, 10)

FEATURE PIPELINE:
6 POLLUTANTS + 5 WEATHER + 10 TEMPORAL = 21 CHANNELS

WEATHER-INTEGRATED INPUT IS READY


In [49]:
# ============================================================
# PHASE 8B.4: ACTIVATE WEATHER-INTEGRATED INPUT
# ============================================================

X = X_weather
Y_reg = Y_reg_weather
Y_cls = Y_cls_weather

P = X.shape[2]
M = X.shape[1]

print("=" * 70)
print("WEATHER INPUT ACTIVATED")
print("=" * 70)

print("X shape:", X.shape)
print("Y_reg shape:", Y_reg.shape)
print("Y_cls shape:", Y_cls.shape)

print("P:", P)
print("M:", M)

assert P == 21
assert M == 10

print()
print("SUCCESS: ST-MPFT INPUT IS NOW 21 CHANNELS")
print("=" * 70)

WEATHER INPUT ACTIVATED
X shape: (3287, 10, 21)
Y_reg shape: (3287, 10)
Y_cls shape: (3287, 10)
P: 21
M: 10

SUCCESS: ST-MPFT INPUT IS NOW 21 CHANNELS


In [50]:
# ============================================================
# PHASE 8B.5: REBUILD 21-CHANNEL NORMALIZED ANCHOR PIPELINE
# ============================================================

print("=" * 70)
print("REBUILDING 21-CHANNEL WEATHER ANCHOR PIPELINE")
print("=" * 70)

# ============================================================
# STEP 1: CHRONOLOGICAL SPLIT
# ============================================================

n_total = len(X)

train_end = int(0.70 * n_total)
val_end = int(0.85 * n_total)

X_train = X[:train_end]
X_val = X[train_end:val_end]
X_test = X[val_end:]

Yr_train = Y_reg[:train_end]
Yr_val = Y_reg[train_end:val_end]
Yr_test = Y_reg[val_end:]

Yc_train_weather = Y_cls[:train_end]
Yc_val_weather = Y_cls[train_end:val_end]
Yc_test_weather = Y_cls[val_end:]

print("Train dates/samples:", len(X_train))
print("Validation dates/samples:", len(X_val))
print("Test dates/samples:", len(X_test))


# ============================================================
# STEP 2: FEATURE NORMALIZATION
# TRAIN STATISTICS ONLY — NO VALIDATION/TEST LEAKAGE
# ============================================================

feature_mu_weather = X_train.mean(
    axis=(0, 1),
    keepdims=True
)

feature_sigma_weather = X_train.std(
    axis=(0, 1),
    keepdims=True
)

feature_sigma_weather[
    feature_sigma_weather < 1e-6
] = 1.0

X_train_n = (
    X_train - feature_mu_weather
) / feature_sigma_weather

X_val_n = (
    X_val - feature_mu_weather
) / feature_sigma_weather

X_test_n = (
    X_test - feature_mu_weather
) / feature_sigma_weather


# ============================================================
# STEP 3: AQI NORMALIZATION
# PER-CITY TRAIN STATISTICS ONLY
# ============================================================

aqi_mu_per_city = Yr_train.mean(
    axis=0,
    keepdims=True
)

aqi_sigma_per_city = Yr_train.std(
    axis=0,
    keepdims=True
)

aqi_sigma_per_city[
    aqi_sigma_per_city < 1e-6
] = 1.0

Yr_train_n = (
    Yr_train - aqi_mu_per_city
) / aqi_sigma_per_city

Yr_val_n = (
    Yr_val - aqi_mu_per_city
) / aqi_sigma_per_city

Yr_test_n = (
    Yr_test - aqi_mu_per_city
) / aqi_sigma_per_city


# ============================================================
# STEP 4: REBUILD ANCHOR WINDOWS
# ============================================================

Xtr_a, Yrtr_a, Yctr_a, Anctr = make_windows_anchor(
    X_train_n,
    Yr_train_n,
    Yc_train_weather,
    H,
    F_HORIZON,
    anchor_window=3
)

Xva_a, Yrva_a, Ycva_a, Ancva = make_windows_anchor(
    X_val_n,
    Yr_val_n,
    Yc_val_weather,
    H,
    F_HORIZON,
    anchor_window=3
)

Xte_a, Yrte_a, Ycte_a, Ancte = make_windows_anchor(
    X_test_n,
    Yr_test_n,
    Yc_test_weather,
    H,
    F_HORIZON,
    anchor_window=3
)


# ============================================================
# STEP 5: RECREATE DATASETS
# ============================================================

train_ds_a = AQIDatasetAnchor(
    Xtr_a,
    Yrtr_a,
    Yctr_a,
    Anctr
)

val_ds_a = AQIDatasetAnchor(
    Xva_a,
    Yrva_a,
    Ycva_a,
    Ancva
)

test_ds_a = AQIDatasetAnchor(
    Xte_a,
    Yrte_a,
    Ycte_a,
    Ancte
)


# ============================================================
# STEP 6: RECREATE DATALOADERS
# ============================================================

train_loader_a = DataLoader(
    train_ds_a,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

val_loader_a = DataLoader(
    val_ds_a,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader_a = DataLoader(
    test_ds_a,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# ============================================================
# STEP 7: FINAL VERIFICATION
# ============================================================

xb_check, yr_check, yc_check, anchor_check = next(
    iter(train_loader_a)
)

INPUT_P = xb_check.shape[-1]

print()
print("=" * 70)
print("21-CHANNEL PIPELINE VERIFICATION")
print("=" * 70)

print("P:", P)
print("INPUT_P:", INPUT_P)

print("X_train_n:", X_train_n.shape)
print("Xtr_a:", Xtr_a.shape)

print("Batch X:", xb_check.shape)
print("Batch regression target:", yr_check.shape)
print("Batch classification target:", yc_check.shape)
print("Batch anchor:", anchor_check.shape)

assert P == 21
assert INPUT_P == 21
assert Xtr_a.shape[-1] == 21

print()
print("SUCCESS: 21 -> 21 -> 21")
print("WEATHER ANCHOR PIPELINE READY")
print("=" * 70)

REBUILDING 21-CHANNEL WEATHER ANCHOR PIPELINE
Train dates/samples: 2300
Validation dates/samples: 493
Test dates/samples: 494

21-CHANNEL PIPELINE VERIFICATION
P: 21
INPUT_P: 21
X_train_n: (2300, 10, 21)
Xtr_a: (2286, 14, 10, 21)
Batch X: torch.Size([16, 14, 10, 21])
Batch regression target: torch.Size([16, 1, 10])
Batch classification target: torch.Size([16, 1, 10])
Batch anchor: torch.Size([16, 10])

SUCCESS: 21 -> 21 -> 21
WEATHER ANCHOR PIPELINE READY


In [52]:
# ============================================================
# PHASE 8B.6: 15-EPOCH WEATHER-INTEGRATED ST-MPFT PROOF RUN
# ============================================================

PROOF_EPOCHS = 15
LAMBDA1 = best_params["lambda1"]
LAMBDA2 = best_params["lambda2"]

print("=" * 70)
print("WEATHER-INTEGRATED ST-MPFT PROOF RUN")
print("=" * 70)

print("Input channels:", INPUT_P)
print("Proof epochs:", PROOF_EPOCHS)

# ============================================================
# USE BEST OPTUNA PARAMS FROM 16-CHANNEL EXPERIMENT
# AS INITIAL PROOF CONFIGURATION
# ============================================================

proof_model = ST_MPFT(
    P=INPUT_P,
    M=M,
    T=H,
    d_model=best_params["d_model"],
    F_horizon=F_HORIZON,
    n_classes=N_CLASSES,
    patch_len=best_params["patch_len"],
    n_heads=best_params["n_heads"],
    use_spatial=True,
    use_chem_attn=True,
    correlation_adj=correlation_adj,
    dropout=best_params["dropout"]
).to(device)

proof_optimizer = torch.optim.AdamW(
    proof_model.parameters(),
    lr=best_params["lr"],
    weight_decay=best_params["weight_decay"]
)

proof_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    proof_optimizer,
    T_max=PROOF_EPOCHS,
    eta_min=1e-6
)

proof_history = []

best_proof_mae = float("inf")


# ============================================================
# TRAINING
# ============================================================

for epoch in range(PROOF_EPOCHS):

    proof_model.train()

    running_loss = 0.0

    for xb, yrb, ycb, anb in train_loader_a:

        xb = xb.to(device)
        yrb = yrb.to(device)
        ycb = ycb.to(device)
        anb = anb.to(device)

        proof_optimizer.zero_grad()

        reg_out, cls_out, _, _ = proof_model(xb)

        loss, _, _ = multitask_loss_delta(
            reg_out,
            yrb,
            anb,
            cls_out,
            ycb,
            best_params["lambda1"],
            best_params["lambda2"]
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            proof_model.parameters(),
            max_norm=1.0
        )

        proof_optimizer.step()

        running_loss += (
            loss.item() * xb.size(0)
        )

    proof_scheduler.step()

    train_loss = (
        running_loss
        / len(train_loader_a.dataset)
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    val_loss, val_mae, val_rmse, val_smape_, val_f1 = (
        evaluate_anchor(
            val_loader_a,
            proof_model
        )
    )

    proof_history.append(
        {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse,
            "val_smape": val_smape_,
            "val_f1": val_f1
        }
    )

    if val_mae < best_proof_mae:

        best_proof_mae = val_mae

        torch.save(
            proof_model.state_dict(),
            "/kaggle/working/best_weather_proof_st_mpft.pt"
        )

    print(
        f"Epoch {epoch + 1:02d}/{PROOF_EPOCHS} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val MAE {val_mae:.2f} | "
        f"RMSE {val_rmse:.2f} | "
        f"SMAPE {val_smape_:.2f} | "
        f"F1 {val_f1:.3f}"
    )


# ============================================================
# FINAL PROOF RESULT
# ============================================================

proof_df = pd.DataFrame(
    proof_history
)

print()
print("=" * 70)
print("WEATHER PROOF RUN COMPLETE")
print("=" * 70)

print(
    "OLD OPTUNA BEST MAE:",
    study.best_value
)

print(
    "OLD CV MEAN MAE:",
    cv_df["MAE"].mean()
)

print(
    "BEST WEATHER PROOF MAE:",
    best_proof_mae
)

print()

improvement_vs_optuna = (
    study.best_value - best_proof_mae
)

print(
    "IMPROVEMENT VS OLD OPTUNA:",
    improvement_vs_optuna
)

print()

if best_proof_mae <= 20:

    print("TARGET ZONE REACHED: 18-20")

elif best_proof_mae < 23:

    print(
        "STRONG IMPROVEMENT: WEATHER PIPELINE IS PROMISING"
    )

elif best_proof_mae < study.best_value:

    print(
        "MODEST IMPROVEMENT: WEATHER HELPS"
    )

else:

    print(
        "NO IMPROVEMENT: WEATHER PIPELINE DID NOT BEAT OLD MODEL"
    )

print("=" * 70)

WEATHER-INTEGRATED ST-MPFT PROOF RUN
Input channels: 21
Proof epochs: 15
Epoch 01/15 | Train Loss 1.8589 | Val MAE 27.79 | RMSE 34.56 | SMAPE 33.42 | F1 0.174
Epoch 02/15 | Train Loss 1.7478 | Val MAE 27.20 | RMSE 33.64 | SMAPE 32.25 | F1 0.174
Epoch 03/15 | Train Loss 1.7079 | Val MAE 26.81 | RMSE 33.19 | SMAPE 31.89 | F1 0.174
Epoch 04/15 | Train Loss 1.6917 | Val MAE 26.59 | RMSE 32.88 | SMAPE 31.68 | F1 0.174
Epoch 05/15 | Train Loss 1.6818 | Val MAE 26.24 | RMSE 32.60 | SMAPE 31.55 | F1 0.174
Epoch 06/15 | Train Loss 1.6749 | Val MAE 26.13 | RMSE 32.41 | SMAPE 31.40 | F1 0.174
Epoch 07/15 | Train Loss 1.6669 | Val MAE 26.11 | RMSE 32.33 | SMAPE 31.29 | F1 0.174
Epoch 08/15 | Train Loss 1.6613 | Val MAE 26.14 | RMSE 32.38 | SMAPE 31.35 | F1 0.174
Epoch 09/15 | Train Loss 1.6619 | Val MAE 26.00 | RMSE 32.26 | SMAPE 31.24 | F1 0.174
Epoch 10/15 | Train Loss 1.6565 | Val MAE 26.21 | RMSE 32.41 | SMAPE 31.36 | F1 0.174
Epoch 11/15 | Train Loss 1.6528 | Val MAE 26.03 | RMSE 32.23 | SMAP

In [53]:
# ============================================================
# PHASE 9: AQI BASELINE + ANCHOR DIAGNOSTIC
# ============================================================

print("=" * 70)
print("AQI BASELINE + ANCHOR DIAGNOSTIC")
print("=" * 70)


# ============================================================
# HELPER
# ============================================================

def denormalize_aqi(x):

    return (
        x * aqi_sigma_per_city
        + aqi_mu_per_city
    )


# ============================================================
# TEST TARGET
# ============================================================

true_test_n = Yrte_a

true_test = denormalize_aqi(
    true_test_n
)


# ============================================================
# BASELINE 1: 3-DAY MEAN ANCHOR
# ============================================================

anchor_pred_n = Ancte[:, None, :]

anchor_pred = denormalize_aqi(
    anchor_pred_n
)

anchor_mae = mean_absolute_error(
    true_test.flatten(),
    anchor_pred.flatten()
)


# ============================================================
# BASELINE 2: LAST OBSERVED AQI
#
# IMPORTANT:
# AQI is not an X input channel.
# Therefore reconstruct persistence from Yr_test_n.
# ============================================================

persistence_preds = []
persistence_true = []

for t in range(
    H,
    len(Yr_test_n) - F_HORIZON + 1
):

    last_aqi = Yr_test_n[t - 1]

    future_aqi = Yr_test_n[
        t:t + F_HORIZON
    ]

    persistence_preds.append(
        last_aqi
    )

    persistence_true.append(
        future_aqi
    )


persistence_preds = np.stack(
    persistence_preds
).astype(np.float32)

persistence_true = np.stack(
    persistence_true
).astype(np.float32)


if F_HORIZON == 1:

    persistence_preds = (
        persistence_preds[:, None, :]
    )


persistence_pred_abs = denormalize_aqi(
    persistence_preds
)

persistence_true_abs = denormalize_aqi(
    persistence_true
)


persistence_mae = mean_absolute_error(
    persistence_true_abs.flatten(),
    persistence_pred_abs.flatten()
)


# ============================================================
# BASELINE 3: CITY TRAINING MEAN
# ============================================================

mean_pred_n = np.zeros_like(
    true_test_n
)

mean_pred = denormalize_aqi(
    mean_pred_n
)

mean_mae = mean_absolute_error(
    true_test.flatten(),
    mean_pred.flatten()
)


# ============================================================
# WEATHER MODEL
# ============================================================

proof_model.load_state_dict(
    torch.load(
        "/kaggle/working/best_weather_proof_st_mpft.pt",
        map_location=device
    )
)

proof_model.eval()

weather_preds = []
weather_true = []

with torch.no_grad():

    for xb, yrb, ycb, anb in test_loader_a:

        xb = xb.to(device)

        yrb = yrb.to(device)

        anb = anb.to(device)


        reg_out, _, _, _ = proof_model(
            xb
        )


        pred_abs_n = (
            reg_out
            + anb.unsqueeze(1)
        )


        weather_preds.append(
            pred_abs_n.cpu().numpy()
        )

        weather_true.append(
            yrb.cpu().numpy()
        )


weather_preds_n = np.concatenate(
    weather_preds,
    axis=0
)

weather_true_n = np.concatenate(
    weather_true,
    axis=0
)


weather_preds_abs = denormalize_aqi(
    weather_preds_n
)

weather_true_abs = denormalize_aqi(
    weather_true_n
)


weather_mae = mean_absolute_error(
    weather_true_abs.flatten(),
    weather_preds_abs.flatten()
)


# ============================================================
# RESULTS
# ============================================================

baseline_results = pd.DataFrame(
    {
        "Method": [
            "City Training Mean",
            "3-Day AQI Anchor",
            "Last AQI Persistence",
            "Weather ST-MPFT"
        ],

        "MAE": [
            mean_mae,
            anchor_mae,
            persistence_mae,
            weather_mae
        ]
    }
)


baseline_results = baseline_results.sort_values(
    "MAE"
).reset_index(
    drop=True
)


print()

print(
    baseline_results.to_string(
        index=False
    )
)


print()

print("=" * 70)
print("DIAGNOSTIC CONCLUSION")
print("=" * 70)


best_baseline_mae = min(
    anchor_mae,
    persistence_mae,
    mean_mae
)


if persistence_mae <= 20:

    print(
        "CRITICAL FINDING:"
    )

    print(
        "SIMPLE AQI PERSISTENCE ALREADY REACHES <= 20 MAE."
    )

    print(
        "THE ST-MPFT TARGET/DELTA FORMULATION NEEDS REDESIGN."
    )


elif best_baseline_mae < weather_mae:

    print(
        "CRITICAL FINDING:"
    )

    print(
        "A SIMPLE BASELINE BEATS ST-MPFT."
    )

    print(
        "DO NOT TRAIN THE CURRENT MODEL FOR 100 EPOCHS."
    )


else:

    print(
        "ST-MPFT BEATS SIMPLE BASELINES."
    )

    print(
        "THE DATA SIGNAL / FEATURE DESIGN IS THE LIKELY BOTTLENECK."
    )


print("=" * 70)

AQI BASELINE + ANCHOR DIAGNOSTIC

              Method       MAE
  City Training Mean 26.037405
     Weather ST-MPFT 26.393620
    3-Day AQI Anchor 29.766874
Last AQI Persistence 36.329792

DIAGNOSTIC CONCLUSION
CRITICAL FINDING:
A SIMPLE BASELINE BEATS ST-MPFT.
DO NOT TRAIN THE CURRENT MODEL FOR 100 EPOCHS.


In [56]:
# ============================================================
# PHASE 10: DIRECT AQI REGRESSION PROOF RUN
# ST-MPFT V2 + WEATHER
# NO ANCHOR / NO CLASSIFICATION GRADIENT
# WEIGHTED HUBER + MAE + MSE
# ============================================================

PROOF_EPOCHS_DIRECT = 25

print("=" * 70)
print("DIRECT AQI ST-MPFT PROOF RUN")
print("=" * 70)

print("Input channels:", INPUT_P)
print("Epochs:", PROOF_EPOCHS_DIRECT)


# ============================================================
# STEP 1: DIRECT AQI LOSS
# ============================================================

def direct_aqi_loss(
    pred,
    target
):

    # --------------------------------------------------------
    # AQI-SEVERITY WEIGHT
    # Higher AQI observations receive greater importance
    # --------------------------------------------------------

    target_abs = (
        target
        * torch.as_tensor(
            aqi_sigma_per_city,
            dtype=target.dtype,
            device=target.device
        )
        + torch.as_tensor(
            aqi_mu_per_city,
            dtype=target.dtype,
            device=target.device
        )
    )


    weights = torch.ones_like(
        target_abs
    )


    weights = torch.where(
        target_abs >= 200,
        1.50,
        weights
    )


    weights = torch.where(
        target_abs >= 300,
        2.00,
        weights
    )


    # --------------------------------------------------------
    # HUBER
    # --------------------------------------------------------

    huber_element = torch.nn.functional.huber_loss(
        pred,
        target,
        delta=1.0,
        reduction="none"
    )


    weighted_huber = (
        huber_element * weights
    ).mean()


    # --------------------------------------------------------
    # MAE
    # --------------------------------------------------------

    mae_loss = torch.mean(
        torch.abs(
            pred - target
        )
    )


    # --------------------------------------------------------
    # MSE
    # --------------------------------------------------------

    mse_loss = torch.mean(
        (
            pred - target
        ) ** 2
    )


    # --------------------------------------------------------
    # HYBRID LOSS
    # --------------------------------------------------------

    total_loss = (
        0.60 * weighted_huber
        + 0.30 * mae_loss
        + 0.10 * mse_loss
    )


    return total_loss


# ============================================================
# STEP 2: DIRECT AQI EVALUATION
# ============================================================

def evaluate_direct_aqi(
    loader,
    model_
):

    model_.eval()


    all_pred = []

    all_true = []


    with torch.no_grad():

        for xb, yrb, ycb, anb in loader:

            xb = xb.to(device)

            yrb = yrb.to(device)


            reg_out, _, _, _ = model_(
                xb
            )


            # IMPORTANT:
            # NO ANCHOR ADDITION
            pred_direct = reg_out


            all_pred.append(
                pred_direct
                .cpu()
                .numpy()
            )


            all_true.append(
                yrb
                .cpu()
                .numpy()
            )


    pred_n = np.concatenate(
        all_pred,
        axis=0
    )


    true_n = np.concatenate(
        all_true,
        axis=0
    )


    pred_abs = (
        pred_n
        * aqi_sigma_per_city
        + aqi_mu_per_city
    )


    true_abs = (
        true_n
        * aqi_sigma_per_city
        + aqi_mu_per_city
    )


    mae = mean_absolute_error(
        true_abs.flatten(),
        pred_abs.flatten()
    )


    rmse = np.sqrt(
        mean_squared_error(
            true_abs.flatten(),
            pred_abs.flatten()
        )
    )


    smape_value = (
        100
        * np.mean(
            2
            * np.abs(
                pred_abs - true_abs
            )
            / (
                np.abs(pred_abs)
                + np.abs(true_abs)
                + 1e-8
            )
        )
    )


    return (
        mae,
        rmse,
        smape_value
    )


# ============================================================
# STEP 3: CREATE FRESH ST-MPFT MODEL
# ============================================================

direct_model = ST_MPFT(
    P=INPUT_P,
    M=M,
    T=H,
    d_model=best_params["d_model"],
    F_horizon=F_HORIZON,
    n_classes=N_CLASSES,
    patch_len=best_params["patch_len"],
    n_heads=best_params["n_heads"],
    use_spatial=True,
    use_chem_attn=True,
    correlation_adj=correlation_adj,
    dropout=best_params["dropout"]
).to(device)


# ============================================================
# STEP 4: OPTIMIZER
# ============================================================

direct_optimizer = torch.optim.AdamW(
    direct_model.parameters(),
    lr=best_params["lr"],
    weight_decay=best_params["weight_decay"]
)


direct_scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        direct_optimizer,
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )
)


# ============================================================
# STEP 5: TRAINING VARIABLES
# ============================================================

best_direct_mae = float("inf")

direct_history = []

patience_direct = 7

counter_direct = 0


# ============================================================
# STEP 6: TRAIN DIRECT AQI MODEL
# ============================================================

for epoch in range(
    PROOF_EPOCHS_DIRECT
):

    direct_model.train()


    running_loss = 0.0


    for xb, yrb, ycb, anb in train_loader_a:

        xb = xb.to(device)

        yrb = yrb.to(device)


        direct_optimizer.zero_grad()


        reg_out, _, _, _ = direct_model(
            xb
        )


        # ====================================================
        # DIRECT AQI REGRESSION
        # NO ANCHOR
        # NO CLASSIFICATION LOSS
        # ====================================================

        loss = direct_aqi_loss(
            reg_out,
            yrb
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            direct_model.parameters(),
            max_norm=1.0
        )


        direct_optimizer.step()


        running_loss += (
            loss.item()
            * xb.size(0)
        )


    train_loss = (
        running_loss
        / len(train_loader_a.dataset)
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    (
        val_mae,
        val_rmse,
        val_smape
    ) = evaluate_direct_aqi(
        val_loader_a,
        direct_model
    )


    direct_scheduler.step(
        val_mae
    )


    current_lr = (
        direct_optimizer
        .param_groups[0]["lr"]
    )


    direct_history.append(
        {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse,
            "val_smape": val_smape,
            "lr": current_lr
        }
    )


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_mae < best_direct_mae:

        best_direct_mae = val_mae

        counter_direct = 0


        torch.save(
            direct_model.state_dict(),
            "/kaggle/working/best_direct_st_mpft.pt"
        )


    else:

        counter_direct += 1


    print(
        f"Epoch {epoch + 1:02d}/"
        f"{PROOF_EPOCHS_DIRECT} | "
        f"Loss {train_loss:.4f} | "
        f"Val MAE {val_mae:.2f} | "
        f"RMSE {val_rmse:.2f} | "
        f"SMAPE {val_smape:.2f} | "
        f"LR {current_lr:.2e}"
    )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if counter_direct >= patience_direct:

        print()

        print(
            "EARLY STOPPING AT EPOCH",
            epoch + 1
        )

        break


# ============================================================
# STEP 7: FINAL RESULT
# ============================================================

direct_df = pd.DataFrame(
    direct_history
)


print()

print("=" * 70)

print(
    "DIRECT AQI PROOF COMPLETE"
)

print("=" * 70)


print(
    "CITY MEAN BASELINE MAE:",
    mean_mae
)


print(
    "OLD OPTUNA BEST MAE:",
    study.best_value
)


print(
    "WEATHER ANCHOR MAE:",
    best_proof_mae
)


print(
    "BEST DIRECT ST-MPFT MAE:",
    best_direct_mae
)


print()


if best_direct_mae <= 20:

    print(
        "TARGET ZONE REACHED: MAE <= 20"
    )


elif best_direct_mae <= 22:

    print(
        "VERY STRONG RESULT: CLOSE TO TARGET"
    )


elif best_direct_mae < 25:

    print(
        "MEANINGFUL IMPROVEMENT: DIRECT REGRESSION WORKS"
    )


elif best_direct_mae < mean_mae:

    print(
        "ST-MPFT NOW BEATS CITY MEAN BASELINE"
    )


else:

    print(
        "DIRECT ST-MPFT STILL FAILS TO BEAT BASELINE"
    )


print("=" * 70)

DIRECT AQI ST-MPFT PROOF RUN
Input channels: 21
Epochs: 25
Epoch 01/25 | Loss 0.6025 | Val MAE 25.63 | RMSE 31.75 | SMAPE 30.80 | LR 3.25e-05
Epoch 02/25 | Loss 0.5994 | Val MAE 25.65 | RMSE 31.76 | SMAPE 30.83 | LR 3.25e-05
Epoch 03/25 | Loss 0.6000 | Val MAE 25.62 | RMSE 31.75 | SMAPE 30.81 | LR 3.25e-05
Epoch 04/25 | Loss 0.5985 | Val MAE 25.62 | RMSE 31.76 | SMAPE 30.83 | LR 3.25e-05
Epoch 05/25 | Loss 0.5996 | Val MAE 25.62 | RMSE 31.75 | SMAPE 30.82 | LR 3.25e-05
Epoch 06/25 | Loss 0.5990 | Val MAE 25.63 | RMSE 31.78 | SMAPE 30.85 | LR 3.25e-05
Epoch 07/25 | Loss 0.5979 | Val MAE 25.62 | RMSE 31.75 | SMAPE 30.82 | LR 3.25e-05
Epoch 08/25 | Loss 0.5993 | Val MAE 25.67 | RMSE 31.78 | SMAPE 30.84 | LR 3.25e-05
Epoch 09/25 | Loss 0.5996 | Val MAE 25.62 | RMSE 31.77 | SMAPE 30.84 | LR 3.25e-05
Epoch 10/25 | Loss 0.5991 | Val MAE 25.68 | RMSE 31.79 | SMAPE 30.84 | LR 3.25e-05
Epoch 11/25 | Loss 0.5984 | Val MAE 25.62 | RMSE 31.76 | SMAPE 30.82 | LR 1.62e-05
Epoch 12/25 | Loss 0.5975 | 

In [19]:

DATA_PATH = '/kaggle/input/datasets/tushardobal/india-city-air-quality-index-dataset-20152023/india_city_aqi_2015_2023.csv'

df = pd.read_csv(DATA_PATH)

# --- Rename to match the rest of the notebook ---
df = df.rename(columns={
    'city': 'City',
    'date': 'Date',
    'pm25': 'PM2.5',
    'pm10': 'PM10',
    'no2': 'NO2',
    'so2': 'SO2',
    'co': 'CO',
    'o3': 'O3',
    'aqi': 'AQI',
    'aqi_category': 'AQI_Bucket'
})

print(df.shape)
df.head()

REQUIRED_COLS = ['City', 'Date', 'PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'AQI', 'AQI_Bucket']
missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    print('WARNING - missing expected columns:', missing)
    print('Available columns:', list(df.columns))
else:
    print('All required columns present ✓')

print('\nUnique AQI_Bucket values:', df['AQI_Bucket'].unique())

(32870, 10)
All required columns present ✓

Unique AQI_Bucket values: ['Moderate' 'Satisfactory' 'Good' 'Poor']


In [20]:
# Preprocessing: missing-value thresholding + imputation (Phase 1 of roadmap)
POLLUTANTS = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3']

df['Date'] = pd.to_datetime(df['Date'])

# --- Missing Value Thresholding: drop stations with >30% missing pollutant records ---
missing_frac = df.groupby('City')[POLLUTANTS].apply(lambda x: x.isna().mean().mean())
valid_cities = missing_frac[missing_frac <= 0.30].index.tolist()
print(f'Kept {len(valid_cities)} / {df.City.nunique()} cities after 30% missingness filter')

df = df[df['City'].isin(valid_cities)].copy()

# --- AQI category -> integer class ---
# NOTE: standard CPCB buckets are 6 (Good..Severe), but this dataset only contains 4.
# We build CATEGORY_ORDER dynamically from whatever categories actually exist in the data,
# in the correct severity order, so nothing breaks if your dataset has a different subset.
FULL_SEVERITY_ORDER = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
present_categories = set(df['AQI_Bucket'].dropna().unique().tolist())
CATEGORY_ORDER = [c for c in FULL_SEVERITY_ORDER if c in present_categories]
N_CLASSES = len(CATEGORY_ORDER)
print('Using', N_CLASSES, 'AQI classes:', CATEGORY_ORDER)

cat_map = {c: i for i, c in enumerate(CATEGORY_ORDER)}
df['AQI_Class'] = df['AQI_Bucket'].map(cat_map)

# --- Pivot into a city x date x pollutant tensor ---
pivot = df.pivot_table(index='Date', columns='City', values=POLLUTANTS + ['AQI', 'AQI_Class'])
pivot = pivot.sort_index()

# --- Spatio-Temporal Imputation (linear interpolation as a practical stand-in for MICE) ---
pivot = pivot.interpolate(method='linear', limit_direction='both')
pivot = pivot.ffill().bfill()

cities = sorted(valid_cities)
M = len(cities)          # number of monitoring stations/nodes
P = len(POLLUTANTS)      # number of pollutants
print('Cities (M):', M, '| Pollutants (P):', P, '| Days:', len(pivot))

Kept 10 / 10 cities after 30% missingness filter
Using 4 AQI classes: ['Good', 'Satisfactory', 'Moderate', 'Poor']
Cities (M): 10 | Pollutants (P): 6 | Days: 3287


In [21]:
# Build tensors  X: [T, M, P]   AQI: [T, M]   Class: [T, M]

T_total = len(pivot)

# ============================================================
# 1. ORIGINAL POLLUTANT FEATURES
# ============================================================

X_pollutants = np.zeros(
    (T_total, M, len(POLLUTANTS)),
    dtype=np.float32
)

for pi, pol in enumerate(POLLUTANTS):
    X_pollutants[:, :, pi] = pivot[pol][cities].values


# ============================================================
# 2. TEMPORAL EMBEDDINGS
# ============================================================

dow = pivot.index.dayofweek.values
month = pivot.index.month.values
day_of_year = pivot.index.dayofyear.values


# Day-of-week cyclical embedding

dow_sin = np.sin(2 * np.pi * dow / 7)
dow_cos = np.cos(2 * np.pi * dow / 7)


# Month cyclical embedding

month_sin = np.sin(2 * np.pi * (month - 1) / 12)
month_cos = np.cos(2 * np.pi * (month - 1) / 12)


# Annual seasonal cycle

year_sin = np.sin(2 * np.pi * day_of_year / 365.25)
year_cos = np.cos(2 * np.pi * day_of_year / 365.25)


# ============================================================
# 3. EXPLICIT INDIA SEASON FEATURES
# ============================================================

# Winter      : December-February
# Pre-monsoon : March-May
# Monsoon     : June-September
# Post-monsoon: October-November

season = np.zeros(T_total, dtype=np.int64)

season[np.isin(month, [12, 1, 2])] = 0
season[np.isin(month, [3, 4, 5])] = 1
season[np.isin(month, [6, 7, 8, 9])] = 2
season[np.isin(month, [10, 11])] = 3

season_onehot = np.eye(4, dtype=np.float32)[season]


# ============================================================
# 4. COMBINE TEMPORAL FEATURES
# ============================================================

temporal_features = np.column_stack([
    dow_sin,
    dow_cos,
    month_sin,
    month_cos,
    year_sin,
    year_cos,
    season_onehot
]).astype(np.float32)


# Repeat temporal features for every city

temporal_features = np.repeat(
    temporal_features[:, None, :],
    M,
    axis=1
)


# ============================================================
# 5. FINAL INPUT TENSOR
# ============================================================

X = np.concatenate(
    [
        X_pollutants,
        temporal_features
    ],
    axis=2
)

P = X.shape[2]


# ============================================================
# TARGETS
# ============================================================

Y_reg = pivot['AQI'][cities].values.astype(np.float32)

Y_cls = (
    pivot['AQI_Class'][cities]
    .fillna(0)
    .values
    .astype(np.int64)
)

Y_cls = np.clip(
    Y_cls,
    0,
    N_CLASSES - 1
)


print('Pollutant channels:', len(POLLUTANTS))
print('Temporal channels:', temporal_features.shape[2])
print('Total input channels P:', P)

print(
    'X shape:', X.shape,
    '| Y_reg shape:', Y_reg.shape,
    '| Y_cls shape:', Y_cls.shape
)

Pollutant channels: 6
Temporal channels: 10
Total input channels P: 16
X shape: (3287, 10, 16) | Y_reg shape: (3287, 10) | Y_cls shape: (3287, 10)


In [22]:
# --- Robust training: identify and cap extreme outlier readings ---
# Extreme sensor spikes (firework days, sensor faults) can dominate the MSE loss and
# destabilize training. We winsorize (cap) extreme values per pollutant using the
# 1st/99th percentile from the TRAINING period only, rather than deleting rows outright.
train_slice = X[:train_end] if 'train_end' in dir() else X[:int(len(X)*0.7)]

lower = np.percentile(train_slice.reshape(-1, P), 1, axis=0)
upper = np.percentile(train_slice.reshape(-1, P), 99, axis=0)

n_capped = np.sum((X < lower) | (X > upper))
X = np.clip(X, lower, upper)
print(f'Winsorized {n_capped} extreme pollutant readings ({n_capped / X.size * 100:.2f}% of all values) '
      f'using 1st/99th percentile bounds from the training period.')

# Same treatment for the AQI target itself
aqi_lower = np.percentile(Y_reg[:train_end] if 'train_end' in dir() else Y_reg[:int(len(Y_reg)*0.7)], 1)
aqi_upper = np.percentile(Y_reg[:train_end] if 'train_end' in dir() else Y_reg[:int(len(Y_reg)*0.7)], 99)
n_aqi_capped = np.sum((Y_reg < aqi_lower) | (Y_reg > aqi_upper))
Y_reg = np.clip(Y_reg, aqi_lower, aqi_upper)
print(f'Winsorized {n_aqi_capped} extreme AQI values.')

Winsorized 4064 extreme pollutant readings (0.77% of all values) using 1st/99th percentile bounds from the training period.
Winsorized 609 extreme AQI values.


In [23]:
# Windowing + time-based train/val/test split (avoids temporal leakage)
H = 14          # lookback window (days)
F_HORIZON = 1  # forecast horizon (days)

def make_windows(X, Yr, Yc, H, F_HORIZON):
    T = X.shape[0]
    Xs, Yrs, Ycs = [], [], []
    for t in range(H, T - F_HORIZON):
        Xs.append(X[t-H:t])
        Yrs.append(Yr[t:t+F_HORIZON])
        Ycs.append(Yc[t:t+F_HORIZON])
    return np.stack(Xs), np.stack(Yrs), np.stack(Ycs)

n = T_total
train_end = int(n * 0.7)
val_end = int(n * 0.85)

X_train_raw, Yr_train_raw, Yc_train_raw = X[:train_end], Y_reg[:train_end], Y_cls[:train_end]
X_val_raw, Yr_val_raw, Yc_val_raw = X[train_end:val_end], Y_reg[train_end:val_end], Y_cls[train_end:val_end]
X_test_raw, Yr_test_raw, Yc_test_raw = X[val_end:], Y_reg[val_end:], Y_cls[val_end:]

mu_per_city = X_train_raw.mean(axis=0)          # shape [M, P] — one mean PER CITY, not one global mean
sigma_per_city = X_train_raw.std(axis=0) + 1e-6

aqi_mu_per_city = Yr_train_raw.mean(axis=0)     # shape [M] — one AQI mean PER CITY
aqi_sigma_per_city = Yr_train_raw.std(axis=0) + 1e-6

def normalize_per_city(Xr, Yr):
    return (Xr - mu_per_city) / sigma_per_city, (Yr - aqi_mu_per_city) / aqi_sigma_per_city

X_train_n, Yr_train_n = normalize_per_city(X_train_raw, Yr_train_raw)
X_val_n, Yr_val_n = normalize_per_city(X_val_raw, Yr_val_raw)
X_test_n, Yr_test_n = normalize_per_city(X_test_raw, Yr_test_raw)

Xtr, Yrtr, Yctr = make_windows(X_train_n, Yr_train_n, Yc_train_raw, H, F_HORIZON)
Xva, Yrva, Ycva = make_windows(X_val_n, Yr_val_n, Yc_val_raw, H, F_HORIZON)
Xte, Yrte, Ycte = make_windows(X_test_n, Yr_test_n, Yc_test_raw, H, F_HORIZON)

print('Train windows:', Xtr.shape, '| Val windows:', Xva.shape, '| Test windows:', Xte.shape)

Train windows: (2285, 14, 10, 16) | Val windows: (478, 14, 10, 16) | Test windows: (479, 14, 10, 16)


In [24]:
print("CHECK X shape:", X.shape)
print("CHECK P:", P)

print("CHECK X_train_n:", X_train_n.shape)
print("CHECK Xtr:", Xtr.shape)

print("LAST DIM X:", X.shape[-1])
print("LAST DIM X_train_n:", X_train_n.shape[-1])
print("LAST DIM Xtr:", Xtr.shape[-1])

CHECK X shape: (3287, 10, 16)
CHECK P: 16
CHECK X_train_n: (2300, 10, 16)
CHECK Xtr: (2285, 14, 10, 16)
LAST DIM X: 16
LAST DIM X_train_n: 16
LAST DIM Xtr: 16


In [25]:
# Dataset / DataLoader
class AQIDataset(Dataset):
    def __init__(self, X, Yr, Yc):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Yr = torch.tensor(Yr, dtype=torch.float32)
        self.Yc = torch.tensor(Yc, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Yr[idx], self.Yc[idx]

BATCH_SIZE = 16

train_ds = AQIDataset(Xtr, Yrtr, Yctr)
val_ds = AQIDataset(Xva, Yrva, Ycva)
test_ds = AQIDataset(Xte, Yrte, Ycte)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


In [26]:
# ============================================================
# Weighted Hybrid Regression Loss
# Huber + MAE + MSE
# High AQI events receive larger weights
# ============================================================

def multitask_loss_delta(
    reg_pred_delta,
    reg_true_abs,
    anchor,
    cls_pred,
    cls_true,
    lambda1=1.0,
    lambda2=1.0
):

    delta_true = (
        reg_true_abs
        - anchor.unsqueeze(1)
    )


    # ========================================================
    # AQI SEVERITY WEIGHTS
    # normalized AQI target magnitude is used as severity proxy
    # ========================================================

    severity = torch.abs(
        reg_true_abs
    )

    severity_weight = (
        1.0
        + 0.50 * severity
    )

    severity_weight = torch.clamp(
        severity_weight,
        min=1.0,
        max=3.0
    )


    # ========================================================
    # HUBER LOSS
    # ========================================================

    huber = F.smooth_l1_loss(
        reg_pred_delta,
        delta_true,
        beta=0.5,
        reduction='none'
    )

    weighted_huber = (
        huber
        * severity_weight
    ).mean()


    # ========================================================
    # MAE LOSS
    # ========================================================

    mae_loss = (
        torch.abs(
            reg_pred_delta
            - delta_true
        )
        * severity_weight
    ).mean()


    # ========================================================
    # MSE LOSS
    # ========================================================

    mse_loss = (
        (
            reg_pred_delta
            - delta_true
        ) ** 2
        * severity_weight
    ).mean()


    # ========================================================
    # HYBRID REGRESSION LOSS
    # ========================================================

    regression_loss = (
        0.60 * weighted_huber
        +
        0.25 * mae_loss
        +
        0.15 * mse_loss
    )


    # ========================================================
    # CLASSIFICATION LOSS
    # ========================================================

    B, F_h, M, C = cls_pred.shape

    ce = F.cross_entropy(
        cls_pred.reshape(-1, C),
        cls_true.reshape(-1)
    )


    # ========================================================
    # MULTI-TASK LOSS
    # ========================================================

    total = (
        lambda1 * regression_loss
        +
        lambda2 * ce
    )


    return (
        total,
        regression_loss.item(),
        ce.item()
    )


# ============================================================
# Evaluation
# ============================================================

def evaluate_anchor(loader, model_):

    model_.eval()

    all_pred_abs = []
    all_true_abs = []

    all_cls_pred = []
    all_cls_true = []

    total_loss = 0.0


    with torch.no_grad():

        for xb, yrb, ycb, anb in loader:

            xb = xb.to(device)
            yrb = yrb.to(device)
            ycb = ycb.to(device)
            anb = anb.to(device)


            reg_out, cls_out, _, _ = model_(
                xb
            )


            loss, _, _ = multitask_loss_delta(
                reg_out,
                yrb,
                anb,
                cls_out,
                ycb,
                LAMBDA1,
                LAMBDA2
            )


            total_loss += (
                loss.item()
                * xb.size(0)
            )


            pred_abs = (
                reg_out
                + anb.unsqueeze(1)
            )


            all_pred_abs.append(
                pred_abs.cpu().numpy()
            )

            all_true_abs.append(
                yrb.cpu().numpy()
            )


            all_cls_pred.append(
                cls_out.argmax(-1)
                .cpu()
                .numpy()
            )

            all_cls_true.append(
                ycb.cpu().numpy()
            )


    pred_abs_arr = np.concatenate(
        all_pred_abs
    )

    true_abs_arr = np.concatenate(
        all_true_abs
    )


    pred_denorm = (
        pred_abs_arr
        * aqi_sigma_per_city
        + aqi_mu_per_city
    ).flatten()


    true_denorm = (
        true_abs_arr
        * aqi_sigma_per_city
        + aqi_mu_per_city
    ).flatten()


    mae, rmse, sm = regression_metrics(
        true_denorm,
        pred_denorm
    )


    cls_pred = np.concatenate(
        all_cls_pred
    ).flatten()


    cls_true = np.concatenate(
        all_cls_true
    ).flatten()


    f1 = f1_score(
        cls_true,
        cls_pred,
        average='macro'
    )


    avg_loss = (
        total_loss
        / len(loader.dataset)
    )


    return (
        avg_loss,
        mae,
        rmse,
        sm,
        f1
    )

In [27]:
# Shared metric utilities (MAE, RMSE, SMAPE, Macro-F1)
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    denom = np.where(denom == 0, 1e-6, denom)
    return np.mean(np.abs(y_true - y_pred) / denom) * 100

def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    sm = smape(y_true, y_pred)
    return mae, rmse, sm

results_table = []   # collects {'Model': ..., 'MAE': ..., 'RMSE': ..., 'SMAPE': ..., 'MacroF1': ...}


In [28]:
# Baseline 1: VAR (Vector Autoregression) — statistical, linear multi-pollutant trends
# VAR needs a flat multivariate series; we average across cities to keep it tractable,
# then forecast AQI for the test horizon.
var_series = pivot['AQI'][cities].mean(axis=1).values  # city-averaged AQI series
var_train = var_series[:train_end]
var_test_true = var_series[val_end:val_end + len(Xte)]

var_input = pd.DataFrame({'AQI': var_train, 'PM2.5': pivot['PM2.5'][cities].mean(axis=1).values[:train_end]})
var_model = VAR(var_input)
var_fitted = var_model.fit(maxlags=7)

var_preds = []
history_window = var_input.values[-var_fitted.k_ar:]
for _ in range(len(Xte)):
    fc = var_fitted.forecast(history_window, steps=F_HORIZON)
    var_preds.append(fc[:, 0])  # AQI column
    history_window = np.vstack([history_window[1:], fc[0]])
var_preds = np.array(var_preds)  # [n_test, F]

var_true_repeated = np.array([var_test_true[i:i+F_HORIZON] if i+F_HORIZON <= len(var_test_true)
                               else np.pad(var_test_true[i:], (0, F_HORIZON-len(var_test_true[i:])), mode='edge')
                               for i in range(len(var_preds))])

mae, rmse, sm = regression_metrics(var_true_repeated.flatten(), var_preds.flatten())
results_table.append({'Model': 'VAR', 'MAE': mae, 'RMSE': rmse, 'SMAPE': sm, 'MacroF1': np.nan})
print(f'VAR  | MAE {mae:.2f} | RMSE {rmse:.2f} | SMAPE {sm:.2f}')


VAR  | MAE 8.05 | RMSE 10.10 | SMAPE 9.21


In [29]:
# Baseline 2: XGBoost — non-linear local point features
# Flatten each [H, M, P] window into a feature vector; predict AQI for a representative city
# (city-averaged) across the F-step horizon.
city_idx_for_baseline = 0  # pick first city as representative point-forecast target

Xtr_flat = Xtr.reshape(len(Xtr), -1)
Xte_flat = Xte.reshape(len(Xte), -1)
Ytr_city = Yrtr[:, :, city_idx_for_baseline]
Yte_city = Yrte[:, :, city_idx_for_baseline]

xgb_model = MultiOutputRegressor(xgb.XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05,
                                                    n_jobs=-1, random_state=SEED))
xgb_model.fit(Xtr_flat, Ytr_city)
xgb_pred = xgb_model.predict(Xte_flat)

xgb_pred_denorm = xgb_pred * aqi_sigma_per_city[city_idx_for_baseline] + aqi_mu_per_city[city_idx_for_baseline]
xgb_true_denorm = Yte_city * aqi_sigma_per_city[city_idx_for_baseline] + aqi_mu_per_city[city_idx_for_baseline]

mae, rmse, sm = regression_metrics(xgb_true_denorm.flatten(), xgb_pred_denorm.flatten())
results_table.append({'Model': 'XGBoost', 'MAE': mae, 'RMSE': rmse, 'SMAPE': sm, 'MacroF1': np.nan})
print(f'XGBoost | MAE {mae:.2f} | RMSE {rmse:.2f} | SMAPE {sm:.2f}')


XGBoost | MAE 25.75 | RMSE 31.79 | SMAPE 30.32


In [30]:
# Baseline 3: Bidirectional LSTM with Attention — sequential dependencies
class BiLSTMAttention(nn.Module):
    def __init__(self, P, M, hidden=64, F_horizon=3, n_classes=6):
        super().__init__()
        self.input_proj = nn.Linear(M * P, hidden)
        self.lstm = nn.LSTM(hidden, hidden, batch_first=True, bidirectional=True)
        self.attn_score = nn.Linear(hidden * 2, 1)
        self.reg_head = nn.Linear(hidden * 2, F_horizon * M)
        self.cls_head = nn.Linear(hidden * 2, F_horizon * M * n_classes)
        self.F_horizon, self.M, self.n_classes = F_horizon, M, n_classes

    def forward(self, x):
        B, T, M, P = x.shape
        x = x.reshape(B, T, M * P)
        x = self.input_proj(x)
        out, _ = self.lstm(x)                       # [B, T, 2*hidden]
        scores = F.softmax(self.attn_score(out), dim=1)
        context = (scores * out).sum(dim=1)          # [B, 2*hidden]
        reg = self.reg_head(context).reshape(B, self.F_horizon, self.M)
        cls = self.cls_head(context).reshape(B, self.F_horizon, self.M, self.n_classes)
        return reg, cls

def train_simple_model(model, train_loader, val_loader, n_epochs, lr=1e-3, lambda1=1.0, lambda2=1.0, tag='model'):
    model = model.to(device)
    opt = torch.optim.AdamW (
        model.parameters(),
        lr=lr,
        weight_decay=0.01,
        betas=(0.9, 0.999)
    )
        
    for ep in range(n_epochs):
        model.train()
        for xb, yrb, ycb in train_loader:
            xb, yrb, ycb = xb.to(device), yrb.to(device), ycb.to(device)
            opt.zero_grad()
            reg_out, cls_out = model(xb)
            mse = F.mse_loss(reg_out, yrb)
            ce = F.cross_entropy(cls_out.reshape(-1, cls_out.shape[-1]), ycb.reshape(-1))
            loss = lambda1 * mse + lambda2 * ce
            loss.backward()
            opt.step()
    return model

def eval_simple_model(model, loader):
    model.eval()
    all_reg_pred, all_reg_true, all_cls_pred, all_cls_true = [], [], [], []
    with torch.no_grad():
        for xb, yrb, ycb in loader:
            xb = xb.to(device)
            reg_out, cls_out = model(xb)
            all_reg_pred.append(reg_out.cpu().numpy())
            all_reg_true.append(yrb.numpy())
            all_cls_pred.append(cls_out.argmax(-1).cpu().numpy())
            all_cls_true.append(ycb.numpy())
    reg_pred = (np.concatenate(all_reg_pred) * aqi_sigma_per_city + aqi_mu_per_city).flatten()
    reg_true = (np.concatenate(all_reg_true) * aqi_sigma_per_city + aqi_mu_per_city).flatten()
    cls_pred = np.concatenate(all_cls_pred).flatten()
    cls_true = np.concatenate(all_cls_true).flatten()
    mae, rmse, sm = regression_metrics(reg_true, reg_pred)
    f1 = f1_score(cls_true, cls_pred, average='macro')
    return mae, rmse, sm, f1

bilstm_model = BiLSTMAttention(P, M, hidden=64, F_horizon=F_HORIZON, n_classes=N_CLASSES)
bilstm_model = train_simple_model(bilstm_model, train_loader, val_loader, n_epochs=30, tag='bilstm')
mae, rmse, sm, f1 = eval_simple_model(bilstm_model, test_loader)
results_table.append({'Model': 'BiLSTM+Attention', 'MAE': mae, 'RMSE': rmse, 'SMAPE': sm, 'MacroF1': f1})
print(f'BiLSTM+Attention | MAE {mae:.2f} | RMSE {rmse:.2f} | SMAPE {sm:.2f} | F1 {f1:.3f}')


BiLSTM+Attention | MAE 29.95 | RMSE 37.12 | SMAPE 36.01 | F1 0.236


In [31]:
# Baseline 4: Simplified STGCN — spatial graph + temporal convolution (no attention)
# Reuses the adaptive-adjacency idea but replaces the transformer with a plain 1D temporal conv,
# matching the "Spatio-Temporal GNN" family (STGCN/DCRNN) in the roadmap's baseline taxonomy.
class SimplifiedSTGCN(nn.Module):
    def __init__(self, P, M, d_model=32, F_horizon=3, n_classes=6, embed_dim=16):
        super().__init__()
        self.linear_in = nn.Linear(P, d_model)
        self.node_embed = nn.Parameter(torch.randn(M, embed_dim) * 0.01)
        self.gcn_weight = nn.Linear(d_model, d_model)
        self.temporal_conv = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1)
        self.reg_head = nn.Linear(d_model, F_horizon)
        self.cls_head = nn.Linear(d_model, F_horizon * n_classes)
        self.F_horizon, self.n_classes = F_horizon, n_classes

    def forward(self, x):
        B, T, M, P = x.shape
        h = F.relu(self.linear_in(x))                      # [B, T, M, d]
        adj = F.softmax(F.relu(self.node_embed @ self.node_embed.T), dim=-1)
        h_flat = h.reshape(B * T, M, -1)
        msg = torch.einsum('mn,bnd->bmd', adj, h_flat).reshape(B, T, M, -1)
        h = F.relu(self.gcn_weight(msg))                     # [B, T, M, d]
        h = h.permute(0, 2, 3, 1).reshape(B * M, -1, T)       # [B*M, d, T]
        h = F.relu(self.temporal_conv(h))
        pooled = h.mean(dim=-1).reshape(B, M, -1)             # temporal pooling
        reg = self.reg_head(pooled).permute(0, 2, 1)
        cls = self.cls_head(pooled).reshape(B, M, self.F_horizon, self.n_classes).permute(0, 2, 1, 3)
        return reg, cls

stgcn_model = SimplifiedSTGCN(P, M, F_horizon=F_HORIZON, n_classes=N_CLASSES)
stgcn_model = train_simple_model(stgcn_model, train_loader, val_loader, n_epochs=30, tag='stgcn')
mae, rmse, sm, f1 = eval_simple_model(stgcn_model, test_loader)
results_table.append({'Model': 'Simplified STGCN', 'MAE': mae, 'RMSE': rmse, 'SMAPE': sm, 'MacroF1': f1})
print(f'Simplified STGCN | MAE {mae:.2f} | RMSE {rmse:.2f} | SMAPE {sm:.2f} | F1 {f1:.3f}')


Simplified STGCN | MAE 25.92 | RMSE 31.73 | SMAPE 31.18 | F1 0.171


In [32]:
# Baseline 5: Simplified PatchTST-style Transformer — long-horizon structural dependencies
# Same temporal patch-transformer used inside ST-MPFT, but with NO chemical attention and
# NO spatial graph — representing the "SOTA pure-transformer" baseline family
# (PatchTST / Informer / iTransformer) from the roadmap.
class PatchTransformerBlock(nn.Module):
    def __init__(self, T, d_model, patch_len=7, n_heads=4, n_layers=2):
        super().__init__()
        self.patch_len = patch_len
        n_patches = max(1, T // patch_len)
        self.n_patches = n_patches
        self.patch_embed = nn.Linear(patch_len * d_model, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches, d_model) * 0.01)
        encoder_layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=d_model*4,
                                                     batch_first=True, dropout=0.1)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, x):
        B, T, M, D = x.shape
        usable_T = self.n_patches * self.patch_len
        x = x[:, :usable_T]
        x = x.permute(0, 2, 1, 3).reshape(B * M, self.n_patches, self.patch_len * D)
        patches = self.patch_embed(x) + self.pos_embed
        encoded = self.transformer(patches)
        pooled = encoded.mean(dim=1).reshape(B, M, D)
        return pooled

class SimplifiedPatchTST(nn.Module):
    def __init__(self, P, M, T, d_model=32, F_horizon=3, n_classes=6):
        super().__init__()
        self.linear_in = nn.Linear(P, d_model)
        self.temporal = PatchTransformerBlock(T, d_model)
        self.reg_head = nn.Linear(d_model, F_horizon)
        self.cls_head = nn.Linear(d_model, F_horizon * n_classes)
        self.F_horizon, self.n_classes = F_horizon, n_classes

    def forward(self, x):
        h = F.relu(self.linear_in(x))
        pooled = self.temporal(h)
        reg = self.reg_head(pooled).permute(0, 2, 1)
        B, M, _ = pooled.shape
        cls = self.cls_head(pooled).reshape(B, M, self.F_horizon, self.n_classes).permute(0, 2, 1, 3)
        return reg, cls

patchtst_model = SimplifiedPatchTST(P, M, H, F_horizon=F_HORIZON, n_classes=N_CLASSES)
patchtst_model = train_simple_model(patchtst_model, train_loader, val_loader, n_epochs=30, tag='patchtst')
mae, rmse, sm, f1 = eval_simple_model(patchtst_model, test_loader)
results_table.append({'Model': 'Simplified PatchTST', 'MAE': mae, 'RMSE': rmse, 'SMAPE': sm, 'MacroF1': f1})
print(f'Simplified PatchTST | MAE {mae:.2f} | RMSE {rmse:.2f} | SMAPE {sm:.2f} | F1 {f1:.3f}')


Simplified PatchTST | MAE 27.24 | RMSE 33.59 | SMAPE 32.61 | F1 0.220


In [33]:
# Phase 2: Proposed ST-MPFT Architecture (Full Model)

In [34]:
# ============================================================
# ST-MPFT V2
# ============================================================


# ============================================================
# BLOCK 1
# FULL MULTI-HEAD CHEMICAL TRANSFORMER ENCODER
# ============================================================

class CrossPollutantAttention(nn.Module):

    def __init__(
        self,
        P,
        d_model,
        n_heads=4,
        dropout=0.1
    ):

        super().__init__()

        self.P = P

        self.proj_in = nn.Linear(
            1,
            d_model
        )

        self.pollutant_embed = nn.Parameter(
            torch.randn(P, d_model) * 0.02
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )

        self.chemical_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        # Separate attention layer only for interpretability
        self.attn_probe = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=0.0,
            batch_first=True
        )

        self.out_pool = nn.Sequential(
            nn.Linear(P * d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model)
        )

        self.final_norm = nn.LayerNorm(d_model)


    def forward(self, x):

        B, T, M, P = x.shape

        x = x.reshape(
            B * T * M,
            P,
            1
        )

        tokens = self.proj_in(x)

        tokens = (
            tokens
            + self.pollutant_embed.unsqueeze(0)
        )

        # Full Transformer Encoder:
        # MH attention + residual + LayerNorm + FFN

        encoded = self.chemical_encoder(
            tokens
        )

        # Attention weights for visualization

        _, attn_weights = self.attn_probe(
            encoded,
            encoded,
            encoded,
            need_weights=True,
            average_attn_weights=True
        )

        flat = encoded.reshape(
            B * T * M,
            P * encoded.shape[-1]
        )

        out = self.out_pool(flat)

        out = self.final_norm(out)

        out = out.reshape(
            B,
            T,
            M,
            -1
        )

        attn_weights = attn_weights.reshape(
            B,
            T,
            M,
            P,
            P
        )

        return out, attn_weights


# ============================================================
# BLOCK 2
# HYBRID ADAPTIVE GRAPH
# Historical Correlation + Learnable Adjacency
# ============================================================

class HybridAdaptiveSpatialGCN(nn.Module):

    def __init__(
        self,
        M,
        d_model,
        correlation_adj,
        embed_dim=16,
        dropout=0.1
    ):

        super().__init__()

        self.M = M

        self.node_embed_1 = nn.Parameter(
            torch.randn(M, embed_dim) * 0.01
        )

        self.node_embed_2 = nn.Parameter(
            torch.randn(M, embed_dim) * 0.01
        )

        self.register_buffer(
            'correlation_adj',
            torch.tensor(
                correlation_adj,
                dtype=torch.float32
            )
        )

        # Learn the contribution of correlation
        # and adaptive graph

        self.graph_logits = nn.Parameter(
            torch.zeros(2)
        )

        self.gcn1 = nn.Linear(
            d_model,
            d_model
        )

        self.gcn2 = nn.Linear(
            d_model,
            d_model
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)


    def forward(self, x):

        B, T, M, D = x.shape


        # ====================================================
        # LEARNABLE ADJACENCY
        # ====================================================

        adaptive_adj = torch.matmul(
            self.node_embed_1,
            self.node_embed_2.T
        )

        adaptive_adj = F.softmax(
            F.relu(adaptive_adj),
            dim=-1
        )


        # ====================================================
        # HISTORICAL CORRELATION GRAPH
        # ====================================================

        corr_adj = self.correlation_adj

        corr_adj = F.softmax(
            corr_adj,
            dim=-1
        )


        # ====================================================
        # HYBRID GRAPH FUSION
        # ====================================================

        graph_weights = F.softmax(
            self.graph_logits,
            dim=0
        )

        hybrid_adj = (
            graph_weights[0] * corr_adj
            +
            graph_weights[1] * adaptive_adj
        )


        # ====================================================
        # RESIDUAL GCN LAYER 1
        # ====================================================

        h = x.reshape(
            B * T,
            M,
            D
        )

        msg1 = torch.einsum(
            'mn,bnd->bmd',
            hybrid_adj,
            h
        )

        msg1 = self.gcn1(msg1)

        msg1 = F.gelu(msg1)

        h = self.norm1(
            h + self.dropout(msg1)
        )


        # ====================================================
        # RESIDUAL GCN LAYER 2
        # ====================================================

        msg2 = torch.einsum(
            'mn,bnd->bmd',
            hybrid_adj,
            h
        )

        msg2 = self.gcn2(msg2)

        msg2 = F.gelu(msg2)

        h = self.norm2(
            h + self.dropout(msg2)
        )


        h = h.reshape(
            B,
            T,
            M,
            D
        )

        return h, hybrid_adj


# ============================================================
# BLOCK 3
# TEMPORAL PATCH TRANSFORMER
# ============================================================

class TemporalPatchTransformer(nn.Module):

    def __init__(
        self,
        T,
        d_model,
        patch_len=7,
        n_heads=4,
        n_layers=3,
        dropout=0.1
    ):

        super().__init__()

        self.patch_len = patch_len

        self.n_patches = max(
            1,
            T // patch_len
        )

        self.patch_embed = nn.Linear(
            patch_len * d_model,
            d_model
        )

        self.pos_embed = nn.Parameter(
            torch.randn(
                1,
                self.n_patches,
                d_model
            ) * 0.01
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.final_norm = nn.LayerNorm(
            d_model
        )


    def forward(self, x):

        B, T, M, D = x.shape

        usable_T = (
            self.n_patches
            * self.patch_len
        )

        # Use the most recent history

        x = x[:, -usable_T:]

        x = x.permute(
            0,
            2,
            1,
            3
        )

        x = x.reshape(
            B * M,
            self.n_patches,
            self.patch_len * D
        )

        patches = self.patch_embed(x)

        patches = (
            patches
            + self.pos_embed
        )

        encoded = self.transformer(
            patches
        )

        encoded = self.final_norm(
            encoded
        )

        # Recent-aware pooling

        pooled = (
            0.7 * encoded[:, -1]
            +
            0.3 * encoded.mean(dim=1)
        )

        pooled = pooled.reshape(
            B,
            M,
            D
        )

        return pooled


# ============================================================
# FULL ST-MPFT V2
# ============================================================

class ST_MPFT(nn.Module):

    def __init__(
        self,
        P,
        M,
        T,
        d_model=64,
        F_horizon=1,
        n_classes=6,
        patch_len=7,
        n_heads=4,
        use_spatial=True,
        use_chem_attn=True,
        correlation_adj=None,
        dropout=0.1
    ):

        super().__init__()

        self.use_spatial = use_spatial
        self.use_chem_attn = use_chem_attn

        self.F_horizon = F_horizon
        self.n_classes = n_classes


        # ====================================================
        # CHEMICAL BLOCK
        # ====================================================

        if use_chem_attn:

            self.chem_attn = CrossPollutantAttention(
                P=P,
                d_model=d_model,
                n_heads=n_heads,
                dropout=dropout
            )

        else:

            self.chem_linear = nn.Sequential(
                nn.Linear(P, d_model),
                nn.GELU(),
                nn.LayerNorm(d_model)
            )


        # ====================================================
        # SPATIAL BLOCK
        # ====================================================

        if use_spatial:

            if correlation_adj is None:

                correlation_adj = np.eye(
                    M,
                    dtype=np.float32
                )

            self.spatial_gcn = HybridAdaptiveSpatialGCN(
                M=M,
                d_model=d_model,
                correlation_adj=correlation_adj,
                embed_dim=16,
                dropout=dropout
            )


        # ====================================================
        # TEMPORAL BLOCK
        # ====================================================

        self.temporal_transformer = TemporalPatchTransformer(
            T=T,
            d_model=d_model,
            patch_len=patch_len,
            n_heads=n_heads,
            n_layers=3,
            dropout=dropout
        )


        # ====================================================
        # FORECAST HEADS
        # ====================================================

        self.reg_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                d_model,
                F_horizon
            )
        )

        self.cls_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                d_model,
                F_horizon * n_classes
            )
        )


    def forward(self, x):

        attn_weights = None
        hybrid_adj = None


        # Chemical Fusion

        if self.use_chem_attn:

            h, attn_weights = self.chem_attn(
                x
            )

        else:

            h = self.chem_linear(
                x
            )


        # Spatial Fusion

        if self.use_spatial:

            h, hybrid_adj = self.spatial_gcn(
                h
            )


        # Temporal Fusion

        pooled = self.temporal_transformer(
            h
        )


        # AQI Regression

        reg_out = self.reg_head(
            pooled
        )

        reg_out = reg_out.permute(
            0,
            2,
            1
        )


        # AQI Classification

        B, M, _ = pooled.shape

        cls_out = self.cls_head(
            pooled
        )

        cls_out = cls_out.reshape(
            B,
            M,
            self.F_horizon,
            self.n_classes
        )

        cls_out = cls_out.permute(
            0,
            2,
            1,
            3
        )


        return (
            reg_out,
            cls_out,
            attn_weights,
            hybrid_adj
        )

In [35]:
# Phase 3a: Hyperparameter Optimization (Optuna)

In [36]:
# ============================================================
# Historical City Correlation Graph
# Training data ONLY - avoids test leakage
# ============================================================

aqi_train_for_graph = Yr_train_raw

correlation_adj = np.corrcoef(
    aqi_train_for_graph.T
)

# Replace NaN / Inf caused by constant series

correlation_adj = np.nan_to_num(
    correlation_adj,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

# Negative correlation is not used as a positive graph edge

correlation_adj = np.maximum(
    correlation_adj,
    0.0
)

# Self connections

np.fill_diagonal(
    correlation_adj,
    1.0
)

correlation_adj = correlation_adj.astype(
    np.float32
)

print(
    'Historical correlation adjacency shape:',
    correlation_adj.shape
)

print(
    'Correlation graph min:',
    correlation_adj.min(),
    '| max:',
    correlation_adj.max(),
    '| mean:',
    correlation_adj.mean()
)

Historical correlation adjacency shape: (10, 10)
Correlation graph min: 0.0 | max: 1.0 | mean: 0.10798281


In [37]:
# ============================================================
# REBUILD ANCHOR PIPELINE USING CURRENT INPUT FEATURES
# FIX: P=16 BUT OLD ANCHOR DATALOADER HAS 10 CHANNELS
# ============================================================

print("=" * 70)
print("REBUILDING ANCHOR PIPELINE")
print("=" * 70)


# ============================================================
# STEP 1: VERIFY CURRENT FEATURE TENSORS
# ============================================================

print("P:", P)
print("X shape:", X.shape)

print("X_train_n shape:", X_train_n.shape)
print("X_val_n shape:", X_val_n.shape)
print("X_test_n shape:", X_test_n.shape)


# ============================================================
# STEP 2: REBUILD CLASSIFICATION TARGET SPLITS
# ============================================================

# Use the same chronological split boundaries already used
# for X and AQI regression targets.

n_total = len(X)

train_end = int(0.70 * n_total)
val_end = int(0.85 * n_total)

Yc_train_fix = Y_cls[:train_end]
Yc_val_fix = Y_cls[train_end:val_end]
Yc_test_fix = Y_cls[val_end:]


print()
print("Classification target split shapes:")

print("Yc_train_fix:", Yc_train_fix.shape)
print("Yc_val_fix:", Yc_val_fix.shape)
print("Yc_test_fix:", Yc_test_fix.shape)


# ============================================================
# STRICT SPLIT ALIGNMENT CHECK
# ============================================================

assert len(X_train_n) == len(Yc_train_fix), (
    f"Train mismatch: X_train_n={len(X_train_n)}, "
    f"Yc_train_fix={len(Yc_train_fix)}"
)

assert len(X_val_n) == len(Yc_val_fix), (
    f"Validation mismatch: X_val_n={len(X_val_n)}, "
    f"Yc_val_fix={len(Yc_val_fix)}"
)

assert len(X_test_n) == len(Yc_test_fix), (
    f"Test mismatch: X_test_n={len(X_test_n)}, "
    f"Yc_test_fix={len(Yc_test_fix)}"
)

print()
print("CLASSIFICATION SPLIT ALIGNMENT PASSED")


# ============================================================
# STEP 3: ANCHOR WINDOW FUNCTION
# ============================================================

def make_windows_anchor(
    X_data,
    Yr_data,
    Yc_data,
    H,
    F_HORIZON,
    anchor_window=3
):

    Xs = []
    Yrs = []
    Ycs = []
    Anchors = []

    total_steps = X_data.shape[0]

    for t in range(
        H,
        total_steps - F_HORIZON + 1
    ):

        # Historical input window

        x_window = X_data[
            t - H:t
        ]


        # Future AQI regression target

        yr_future = Yr_data[
            t:t + F_HORIZON
        ]


        # Future AQI classification target

        yc_future = Yc_data[
            t:t + F_HORIZON
        ]


        # Historical AQI anchor

        anchor_start = max(
            0,
            t - anchor_window
        )

        anchor = Yr_data[
            anchor_start:t
        ].mean(axis=0)


        Xs.append(x_window)
        Yrs.append(yr_future)
        Ycs.append(yc_future)
        Anchors.append(anchor)


    return (
        np.stack(Xs).astype(np.float32),
        np.stack(Yrs).astype(np.float32),
        np.stack(Ycs).astype(np.int64),
        np.stack(Anchors).astype(np.float32)
    )


# ============================================================
# STEP 4: REBUILD TRAIN / VALIDATION / TEST WINDOWS
# ============================================================

Xtr_a, Yrtr_a, Yctr_a, Anctr = make_windows_anchor(
    X_train_n,
    Yr_train_n,
    Yc_train_fix,
    H,
    F_HORIZON,
    anchor_window=3
)


Xva_a, Yrva_a, Ycva_a, Ancva = make_windows_anchor(
    X_val_n,
    Yr_val_n,
    Yc_val_fix,
    H,
    F_HORIZON,
    anchor_window=3
)


Xte_a, Yrte_a, Ycte_a, Ancte = make_windows_anchor(
    X_test_n,
    Yr_test_n,
    Yc_test_fix,
    H,
    F_HORIZON,
    anchor_window=3
)


# ============================================================
# STEP 5: ANCHOR DATASET
# ============================================================

class AQIDatasetAnchor(Dataset):

    def __init__(
        self,
        X,
        Yr,
        Yc,
        Anchor
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.Yr = torch.tensor(
            Yr,
            dtype=torch.float32
        )

        self.Yc = torch.tensor(
            Yc,
            dtype=torch.long
        )

        self.Anchor = torch.tensor(
            Anchor,
            dtype=torch.float32
        )


    def __len__(self):

        return len(self.X)


    def __getitem__(self, idx):

        return (
            self.X[idx],
            self.Yr[idx],
            self.Yc[idx],
            self.Anchor[idx]
        )


# ============================================================
# STEP 6: RECREATE DATASETS
# ============================================================

train_ds_a = AQIDatasetAnchor(
    Xtr_a,
    Yrtr_a,
    Yctr_a,
    Anctr
)


val_ds_a = AQIDatasetAnchor(
    Xva_a,
    Yrva_a,
    Ycva_a,
    Ancva
)


test_ds_a = AQIDatasetAnchor(
    Xte_a,
    Yrte_a,
    Ycte_a,
    Ancte
)


# ============================================================
# STEP 7: RECREATE DATALOADERS
# ============================================================

train_loader_a = DataLoader(
    train_ds_a,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)


val_loader_a = DataLoader(
    val_ds_a,
    batch_size=BATCH_SIZE,
    shuffle=False
)


test_loader_a = DataLoader(
    test_ds_a,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# ============================================================
# STEP 8: FINAL CHANNEL VERIFICATION
# ============================================================

xb_check, yr_check, yc_check, anchor_check = next(
    iter(train_loader_a)
)


INPUT_P = xb_check.shape[-1]


print()
print("=" * 70)
print("FINAL CHANNEL VERIFICATION")
print("=" * 70)

print("P:", P)

print(
    "Raw X channels:",
    X.shape[-1]
)

print(
    "X_train_n channels:",
    X_train_n.shape[-1]
)

print(
    "Xtr_a channels:",
    Xtr_a.shape[-1]
)

print(
    "DataLoader channels:",
    INPUT_P
)

print()
print("DataLoader X shape:", xb_check.shape)
print("Regression target shape:", yr_check.shape)
print("Classification target shape:", yc_check.shape)
print("Anchor shape:", anchor_check.shape)


# ============================================================
# STRICT ASSERTIONS
# ============================================================

assert X.shape[-1] == P, (
    f"X has {X.shape[-1]} channels but P={P}"
)


assert X_train_n.shape[-1] == P, (
    f"X_train_n has {X_train_n.shape[-1]} "
    f"channels but P={P}"
)


assert Xtr_a.shape[-1] == P, (
    f"Xtr_a has {Xtr_a.shape[-1]} "
    f"channels but P={P}"
)


assert INPUT_P == P, (
    f"DataLoader has {INPUT_P} "
    f"channels but P={P}"
)


print()
print("CHANNEL PIPELINE FIX SUCCESSFUL")
print()
print(
    f"COMPLETE PIPELINE: "
    f"{P} -> {P} -> {P} -> {INPUT_P}"
)

print("=" * 70)

REBUILDING ANCHOR PIPELINE
P: 16
X shape: (3287, 10, 16)
X_train_n shape: (2300, 10, 16)
X_val_n shape: (493, 10, 16)
X_test_n shape: (494, 10, 16)

Classification target split shapes:
Yc_train_fix: (2300, 10)
Yc_val_fix: (493, 10)
Yc_test_fix: (494, 10)

CLASSIFICATION SPLIT ALIGNMENT PASSED

FINAL CHANNEL VERIFICATION
P: 16
Raw X channels: 16
X_train_n channels: 16
Xtr_a channels: 16
DataLoader channels: 16

DataLoader X shape: torch.Size([16, 14, 10, 16])
Regression target shape: torch.Size([16, 1, 10])
Classification target shape: torch.Size([16, 1, 10])
Anchor shape: torch.Size([16, 10])

CHANNEL PIPELINE FIX SUCCESSFUL

COMPLETE PIPELINE: 16 -> 16 -> 16 -> 16


In [38]:
# ============================================================
# SYSTEMATIC OPTUNA SEARCH FOR ST-MPFT V2
# ============================================================

N_TRIALS = 40
HPO_EPOCHS = 20

# ============================================================
# INPUT CHANNEL SAFETY CHECK
# ============================================================

xb_check, _, _, _ = next(iter(train_loader_a))

INPUT_P = xb_check.shape[-1]

print("Model P variable:", P)
print("Actual DataLoader channels:", INPUT_P)

assert P == INPUT_P, (
    f"P mismatch detected: P={P}, "
    f"but DataLoader has {INPUT_P} input channels."
)

print("INPUT CHANNEL CHECK PASSED")

def objective(trial):

    # ========================================================
    # STRUCTURAL HYPERPARAMETERS
    # ========================================================

    d_model = trial.suggest_categorical(
        'd_model',
        [32, 64, 96, 128]
    )

    n_heads = trial.suggest_categorical(
        'n_heads',
        [2, 4, 8]
    )


    # Transformer requirement:
    # d_model must be divisible by n_heads

    if d_model % n_heads != 0:

        raise optuna.TrialPruned()


    patch_len = trial.suggest_categorical(
        'patch_len',
        [2, 7, 14]
    )


    # Patch must fit inside H

    if patch_len > H:

        raise optuna.TrialPruned()


    # ========================================================
    # OPTIMIZATION HYPERPARAMETERS
    # ========================================================

    lr = trial.suggest_float(
        'lr',
        1e-5,
        2e-3,
        log=True
    )


    weight_decay = trial.suggest_float(
        'weight_decay',
        1e-6,
        1e-2,
        log=True
    )


    lambda1 = trial.suggest_float(
        'lambda1',
        0.8,
        2.0
    )


    lambda2 = trial.suggest_float(
        'lambda2',
        0.05,
        0.8
    )


    dropout = trial.suggest_float(
        'dropout',
        0.05,
        0.30
    )


    # ========================================================
    # MODEL
    # ========================================================

    trial_model = ST_MPFT(
        P=INPUT_P,
        M=M,
        T=H,
        d_model=d_model,
        F_horizon=F_HORIZON,
        n_classes=N_CLASSES,
        patch_len=patch_len,
        n_heads=n_heads,
        use_spatial=True,
        use_chem_attn=True,
        correlation_adj=correlation_adj,
        dropout=dropout
    ).to(device)


    # ========================================================
    # OPTIMIZER
    # ========================================================

    opt = torch.optim.AdamW(
        trial_model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )


    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=HPO_EPOCHS,
        eta_min=1e-6
    )


    # ========================================================
    # TRAINING
    # ========================================================

    for ep in range(HPO_EPOCHS):

        trial_model.train()


        for xb, yrb, ycb, anb in train_loader_a:

            xb = xb.to(device)
            yrb = yrb.to(device)
            ycb = ycb.to(device)
            anb = anb.to(device)


            opt.zero_grad()


            reg_out, cls_out, _, _ = trial_model(
                xb
            )


            loss, _, _ = multitask_loss_delta(
                reg_out,
                yrb,
                anb,
                cls_out,
                ycb,
                lambda1,
                lambda2
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                trial_model.parameters(),
                max_norm=1.0
            )


            opt.step()


        scheduler.step()


        # ====================================================
        # OPTUNA PRUNING
        # ====================================================

        _, val_mae, _, _, _ = evaluate_anchor_hpo(
            val_loader_a,
            trial_model,
            lambda1,
            lambda2
        )


        trial.report(
            val_mae,
            ep
        )


        if trial.should_prune():

            raise optuna.TrialPruned()


    return val_mae


# ============================================================
# HPO EVALUATION
# ============================================================

def evaluate_anchor_hpo(
    loader,
    model_,
    lambda1,
    lambda2
):

    model_.eval()

    all_pred = []
    all_true = []


    with torch.no_grad():

        for xb, yrb, ycb, anb in loader:

            xb = xb.to(device)
            yrb = yrb.to(device)
            ycb = ycb.to(device)
            anb = anb.to(device)


            reg_out, cls_out, _, _ = model_(
                xb
            )


            pred_abs = (
                reg_out
                + anb.unsqueeze(1)
            )


            all_pred.append(
                pred_abs.cpu().numpy()
            )

            all_true.append(
                yrb.cpu().numpy()
            )


    pred_arr = np.concatenate(
        all_pred
    )


    true_arr = np.concatenate(
        all_true
    )


    pred_denorm = (
        pred_arr
        * aqi_sigma_per_city
        + aqi_mu_per_city
    ).flatten()


    true_denorm = (
        true_arr
        * aqi_sigma_per_city
        + aqi_mu_per_city
    ).flatten()


    mae = mean_absolute_error(
        true_denorm,
        pred_denorm
    )


    return (
        0.0,
        mae,
        0.0,
        0.0,
        0.0
    )


# ============================================================
# RUN OPTUNA
# ============================================================

pruner = optuna.pruners.MedianPruner(
    n_startup_trials=5,
    n_warmup_steps=5
)


study = optuna.create_study(
    direction='minimize',
    pruner=pruner
)


study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)


print(
    '\nBest hyperparameters found:'
)

print(
    study.best_params
)


print(
    '\nBest validation MAE:',
    study.best_value
)


best_params = study.best_params

[I 2026-07-15 17:15:23,301] A new study created in memory with name: no-name-c95e525d-e859-4520-9180-b28b8eb8d239


Model P variable: 16
Actual DataLoader channels: 16
INPUT CHANNEL CHECK PASSED


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-07-15 17:18:31,539] Trial 0 finished with value: 26.090211868286133 and parameters: {'d_model': 64, 'n_heads': 8, 'patch_len': 2, 'lr': 0.00035335630682699527, 'weight_decay': 1.6127378121047887e-05, 'lambda1': 1.0980227471907908, 'lambda2': 0.7987016727721779, 'dropout': 0.22785310712333712}. Best is trial 0 with value: 26.090211868286133.
[I 2026-07-15 17:21:36,384] Trial 1 finished with value: 27.42746925354004 and parameters: {'d_model': 64, 'n_heads': 8, 'patch_len': 14, 'lr': 0.0012454001513178723, 'weight_decay': 0.00015556903044562027, 'lambda1': 1.8618146431680391, 'lambda2': 0.428282330559382, 'dropout': 0.1291454646669461}. Best is trial 0 with value: 26.090211868286133.
[I 2026-07-15 17:23:23,996] Trial 2 finished with value: 26.91126251220703 and parameters: {'d_model': 32, 'n_heads': 4, 'patch_len': 2, 'lr': 2.3078252133278958e-05, 'weight_decay': 5.255811034505024e-06, 'lambda1': 1.3869376682557228, 'lambda2': 0.18352773480658874, 'dropout': 0.14211551423407026}.

In [39]:
# Phase 3b: 5-Fold Rolling-Origin Time-Series Cross-Validation

In [41]:
# ============================================================
# PHASE 3B: 5-FOLD ROLLING-ORIGIN TIME-SERIES CROSS-VALIDATION
# CORRECTED FOR ST-MPFT V2 + ANCHOR DELTA PIPELINE
# ============================================================

N_FOLDS = 5
CV_EPOCHS = 15

fold_size = len(Xtr_a) // (N_FOLDS + 1)

cv_results = []


print("=" * 70)
print("ST-MPFT V2 ROLLING-ORIGIN CROSS-VALIDATION")
print("=" * 70)

print("Total anchor training windows:", len(Xtr_a))
print("Fold size:", fold_size)

print()


# ============================================================
# ROLLING-ORIGIN FOLDS
# ============================================================

for fold in range(N_FOLDS):

    train_end_idx = fold_size * (fold + 1)

    val_start_idx = train_end_idx

    val_end_idx = train_end_idx + fold_size


    if val_end_idx > len(Xtr_a):
        break


    print()
    print("-" * 70)

    print(
        f"FOLD {fold + 1}/{N_FOLDS}"
    )

    print(
        f"Train indices: 0 to {train_end_idx - 1}"
    )

    print(
        f"Validation indices: "
        f"{val_start_idx} to {val_end_idx - 1}"
    )

    print("-" * 70)


    # ========================================================
    # CREATE ANCHOR DATASETS
    # ========================================================

    fold_train_ds = AQIDatasetAnchor(
        Xtr_a[:train_end_idx],
        Yrtr_a[:train_end_idx],
        Yctr_a[:train_end_idx],
        Anctr[:train_end_idx]
    )


    fold_val_ds = AQIDatasetAnchor(
        Xtr_a[val_start_idx:val_end_idx],
        Yrtr_a[val_start_idx:val_end_idx],
        Yctr_a[val_start_idx:val_end_idx],
        Anctr[val_start_idx:val_end_idx]
    )


    # ========================================================
    # DATALOADERS
    # ========================================================

    fold_train_loader = DataLoader(
        fold_train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True
    )


    fold_val_loader = DataLoader(
        fold_val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False
    )


    # ========================================================
    # MODEL
    # ========================================================

    fold_model = ST_MPFT(
        P=INPUT_P,
        M=M,
        T=H,
        d_model=best_params['d_model'],
        F_horizon=F_HORIZON,
        n_classes=N_CLASSES,
        patch_len=best_params['patch_len'],
        n_heads=best_params['n_heads'],
        use_spatial=True,
        use_chem_attn=True,
        correlation_adj=correlation_adj,
        dropout=best_params['dropout']
    ).to(device)


    # ========================================================
    # OPTIMIZER
    # ========================================================

    opt = torch.optim.AdamW(
        fold_model.parameters(),
        lr=best_params['lr'],
        weight_decay=best_params['weight_decay']
    )


    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=CV_EPOCHS,
        eta_min=1e-6
    )


    # ========================================================
    # TRAINING
    # ========================================================

    for ep in range(CV_EPOCHS):

        fold_model.train()

        running_loss = 0.0


        for xb, yrb, ycb, anb in fold_train_loader:

            xb = xb.to(device)

            yrb = yrb.to(device)

            ycb = ycb.to(device)

            anb = anb.to(device)


            opt.zero_grad()


            reg_out, cls_out, _, _ = fold_model(
                xb
            )


            loss, _, _ = multitask_loss_delta(
                reg_out,
                yrb,
                anb,
                cls_out,
                ycb,
                best_params['lambda1'],
                best_params['lambda2']
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                fold_model.parameters(),
                max_norm=1.0
            )


            opt.step()


            running_loss += (
                loss.item() * xb.size(0)
            )


        scheduler.step()


        epoch_loss = (
            running_loss
            / len(fold_train_loader.dataset)
        )


        if (
            (ep + 1) % 5 == 0
            or ep == 0
        ):

            print(
                f"Fold {fold + 1} | "
                f"Epoch {ep + 1}/{CV_EPOCHS} | "
                f"Train Loss {epoch_loss:.4f}"
            )


    # ========================================================
    # VALIDATION
    # ========================================================

    fold_model.eval()


    all_reg_pred = []

    all_reg_true = []

    all_cls_pred = []

    all_cls_true = []


    with torch.no_grad():

        for xb, yrb, ycb, anb in fold_val_loader:

            xb = xb.to(device)

            yrb = yrb.to(device)

            anb = anb.to(device)


            reg_out, cls_out, _, _ = fold_model(
                xb
            )


            # =================================================
            # DELTA -> ABSOLUTE AQI
            # =================================================

            pred_abs = (
                reg_out
                + anb.unsqueeze(1)
            )


            all_reg_pred.append(
                pred_abs.cpu().numpy()
            )


            all_reg_true.append(
                yrb.cpu().numpy()
            )


            all_cls_pred.append(
                cls_out.argmax(-1)
                .cpu()
                .numpy()
            )


            all_cls_true.append(
                ycb.numpy()
            )


    # ========================================================
    # CONCATENATE
    # ========================================================

    reg_pred_n = np.concatenate(
        all_reg_pred,
        axis=0
    )


    reg_true_n = np.concatenate(
        all_reg_true,
        axis=0
    )


    cls_pred = np.concatenate(
        all_cls_pred,
        axis=0
    ).flatten()


    cls_true = np.concatenate(
        all_cls_true,
        axis=0
    ).flatten()


    # ========================================================
    # DENORMALIZE AQI
    # ========================================================

    reg_pred = (
        reg_pred_n
        * aqi_sigma_per_city
        + aqi_mu_per_city
    ).flatten()


    reg_true = (
        reg_true_n
        * aqi_sigma_per_city
        + aqi_mu_per_city
    ).flatten()


    # ========================================================
    # METRICS
    # ========================================================

    mae, rmse, sm = regression_metrics(
        reg_true,
        reg_pred
    )


    f1 = f1_score(
        cls_true,
        cls_pred,
        average='macro'
    )


    cv_results.append(
        {
            'fold': fold + 1,
            'MAE': mae,
            'RMSE': rmse,
            'SMAPE': sm,
            'MacroF1': f1
        }
    )


    print()

    print(
        f"FOLD {fold + 1} RESULT | "
        f"MAE {mae:.2f} | "
        f"RMSE {rmse:.2f} | "
        f"SMAPE {sm:.2f} | "
        f"F1 {f1:.3f}"
    )


    # ========================================================
    # FREE GPU MEMORY
    # ========================================================

    del fold_model

    del opt

    del scheduler

    torch.cuda.empty_cache()


# ============================================================
# CROSS-VALIDATION SUMMARY
# ============================================================

cv_df = pd.DataFrame(
    cv_results
)


print()

print("=" * 70)

print(
    "CROSS-VALIDATION SUMMARY"
)

print("=" * 70)


print(
    cv_df
)


print()

print(
    "MEAN ± STD"
)


print(
    cv_df[
        [
            'MAE',
            'RMSE',
            'SMAPE',
            'MacroF1'
        ]
    ].agg(
        [
            'mean',
            'std'
        ]
    )
)


print("=" * 70)

ST-MPFT V2 ROLLING-ORIGIN CROSS-VALIDATION
Total anchor training windows: 2286
Fold size: 381


----------------------------------------------------------------------
FOLD 1/5
Train indices: 0 to 380
Validation indices: 381 to 761
----------------------------------------------------------------------
Fold 1 | Epoch 1/15 | Train Loss 1.9498
Fold 1 | Epoch 5/15 | Train Loss 1.7514
Fold 1 | Epoch 10/15 | Train Loss 1.6987
Fold 1 | Epoch 15/15 | Train Loss 1.6711

FOLD 1 RESULT | MAE 26.92 | RMSE 33.33 | SMAPE 32.24 | F1 0.227

----------------------------------------------------------------------
FOLD 2/5
Train indices: 0 to 761
Validation indices: 762 to 1142
----------------------------------------------------------------------
Fold 2 | Epoch 1/15 | Train Loss 1.9213
Fold 2 | Epoch 5/15 | Train Loss 1.7274
Fold 2 | Epoch 10/15 | Train Loss 1.6862
Fold 2 | Epoch 15/15 | Train Loss 1.6643

FOLD 2 RESULT | MAE 26.04 | RMSE 32.07 | SMAPE 31.27 | F1 0.174

-----------------------------------

In [ ]:
def multitask_loss_delta(reg_pred_delta, reg_true_abs, anchor, cls_pred, cls_true, lambda1=1.0, lambda2=1.0):
    delta_true = reg_true_abs - anchor.unsqueeze(1)   # broadcast anchor over forecast horizon
    mse = F.mse_loss(reg_pred_delta, delta_true)
    B, F_h, M, C = cls_pred.shape
    ce = F.cross_entropy(cls_pred.reshape(-1, C), cls_true.reshape(-1))
    total = lambda1 * mse + lambda2 * ce
    return total, mse.item(), ce.item()

def evaluate_anchor(loader, model_):
    model_.eval()
    all_pred_abs, all_true_abs, all_cls_pred, all_cls_true = [], [], [], []
    total_loss = 0.0
    with torch.no_grad():
        for xb, yrb, ycb, anb in loader:
            xb, yrb, ycb, anb = xb.to(device), yrb.to(device), ycb.to(device), anb.to(device)
            reg_out, cls_out, _, _ = model_(xb)
            loss, _, _ = multitask_loss_delta(reg_out, yrb, anb, cls_out, ycb, LAMBDA1, LAMBDA2)
            total_loss += loss.item() * xb.size(0)
            pred_abs = reg_out + anb.unsqueeze(1)
            all_pred_abs.append(pred_abs.cpu().numpy())
            all_true_abs.append(yrb.cpu().numpy())
            all_cls_pred.append(cls_out.argmax(-1).cpu().numpy())
            all_cls_true.append(ycb.cpu().numpy())

    pred_abs_arr = np.concatenate(all_pred_abs)        # shape [N, F, M]
    true_abs_arr = np.concatenate(all_true_abs)        # shape [N, F, M]
    # de-normalize using PER-CITY stats (broadcasts correctly over the last dimension M)
    pred_denorm = (pred_abs_arr * aqi_sigma_per_city + aqi_mu_per_city).flatten()
    true_denorm = (true_abs_arr * aqi_sigma_per_city + aqi_mu_per_city).flatten()

    mae, rmse, sm = regression_metrics(true_denorm, pred_denorm)
    cls_pred = np.concatenate(all_cls_pred).flatten()
    cls_true = np.concatenate(all_cls_true).flatten()
    f1 = f1_score(cls_true, cls_pred, average='macro')
    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, mae, rmse, sm, f1

In [ ]:
# Sanity check: naive persistence baseline (predict "no change" from the last known AQI)
anchor_denorm = Ancte * aqi_sigma_per_city + aqi_mu_per_city          # shape [N, M]
true_denorm_full = Yrte * aqi_sigma_per_city + aqi_mu_per_city        # shape [N, F, M]
naive_pred = np.repeat(anchor_denorm[:, None, :], F_HORIZON, axis=1)  # shape [N, F, M]

naive_mae = mean_absolute_error(true_denorm_full.flatten(), naive_pred.flatten())
print(f'Naive persistence MAE (predicting zero change): {naive_mae:.2f}')
print('Your ST-MPFT MAE must beat this number to prove the model is learning anything real.')

In [ ]:
# Phase 3c: Main Training Run — Full 100 Epochs with Live Graphs

In [ ]:
# Final model uses the best hyperparameters found by Optuna, trained for the full 100 epochs
# required by the assignment, with live-updating loss/metric plots saved each epoch.

d_model = best_params['d_model']
model = ST_MPFT(P=P, M=M, T=H, d_model=d_model, F_horizon=F_HORIZON, n_classes=N_CLASSES,
                 patch_len=best_params['patch_len'], n_heads=best_params['n_heads'],
                 use_spatial=True, use_chem_attn=True).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=best_params['lr'], weight_decay=best_params['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=10,
    T_mult=2,
    eta_min=1e-6
)

N_EPOCHS = 100
LAMBDA1, LAMBDA2 = best_params['lambda1'], best_params['lambda2']

history = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'val_rmse': [], 'val_smape': [], 'val_f1': []}


def live_plot(history, epoch):
    clear_output(wait=True)
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['val_loss'], label='Val Loss')
    axes[0].set_title(f'Loss (epoch {epoch+1}/{N_EPOCHS})'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(history['val_mae'], label='Val MAE', color='orange')
    axes[1].plot(history['val_rmse'], label='Val RMSE', color='red')
    axes[1].set_title('Regression Error (AQI units)'); axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(history['val_smape'], label='Val SMAPE (%)', color='purple')
    axes[2].set_title('SMAPE'); axes[2].legend(); axes[2].grid(alpha=0.3)

    axes[3].plot(history['val_f1'], label='Val Macro F1', color='green')
    axes[3].set_title('Classification Macro F1'); axes[3].legend(); axes[3].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('/kaggle/working/figures/training_curves.png', dpi=120)
    plt.show()


best_val_loss_u = float('inf')

UNLEARN_EPOCHS = 100
best_val_loss_u = float('inf')

# ---------- Early Stopping ----------
patience = 15
counter = 0
best_mae = float('inf')
# ------------------------------------

for epoch in range(UNLEARN_EPOCHS):

    model.train()
    running_loss = 0.0

    for xb, yrb, ycb, anb in train_loader_a:

        xb = xb.to(device)
        yrb = yrb.to(device)
        ycb = ycb.to(device)
        anb = anb.to(device)

        optimizer.zero_grad()

        reg_out, cls_out, _, _ = model(xb)

        loss, _, _ = multitask_loss_delta(
            reg_out,
            yrb,
            anb,
            cls_out,
            ycb,
            LAMBDA1,
            LAMBDA2
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += loss.item() * xb.size(0)

    # ---------- End of Training Loop ----------

    train_loss = running_loss / len(train_loader_a.dataset)

    val_loss, val_mae, val_rmse, val_smape_, val_f1 = evaluate_anchor(
        val_loader_a,
        model
    )

    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_mae'].append(val_mae)
    history['val_rmse'].append(val_rmse)
    history['val_smape'].append(val_smape_)
    history['val_f1'].append(val_f1)

    if val_mae < best_mae:

        best_mae = val_mae
        counter = 0

        torch.save(
            model.state_dict(),
            '/kaggle/working/best_st_mpft.pt'
        )

    else:

        counter += 1

    if counter >= patience:
        print(f"Early stopping at Epoch {epoch+1}")
        break

    live_plot(history, epoch)

    print(
        f"Epoch {epoch+1}/{N_EPOCHS} | "
        f"Train Loss {train_loss:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"MAE {val_mae:.2f} | "
        f"RMSE {val_rmse:.2f} | "
        f"SMAPE {val_smape_:.2f} | "
        f"F1 {val_f1:.3f}"
    )

In [ ]:
# Final ST-MPFT test evaluation
model.load_state_dict(torch.load('/kaggle/working/best_st_mpft.pt'))
test_loss, test_mae, test_rmse, test_smape_, test_f1 = evaluate_anchor(test_loader_a, model)
print(f'ST-MPFT TEST | Loss {test_loss:.4f} | MAE {test_mae:.2f} | RMSE {test_rmse:.2f} '
      f'| SMAPE {test_smape_:.2f} | Macro F1 {test_f1:.3f}')
results_table.append({'Model': 'ST-MPFT (proposed)', 'MAE': test_mae, 'RMSE': test_rmse,
                       'SMAPE': test_smape_, 'MacroF1': test_f1})

In [ ]:
# Ablation Study Matrix (actually trained, not just described)

In [ ]:
# Ablation A: No Spatial Graph (identity adjacency -> cities treated independently)
# Ablation B: No Chemical Attention (simple linear layer instead)
# Ablation C: Unweighted Loss (lambda2 = 0, classification gradient removed)

ABLATION_EPOCHS = 60  # reduced from 100 for runtime; raise if time allows

def run_ablation(use_spatial, use_chem_attn, lambda2, tag):

    abl_model = ST_MPFT(
        P, M, H,
        d_model=d_model,
        F_horizon=F_HORIZON,
        n_classes=N_CLASSES,
        patch_len=best_params['patch_len'],
        n_heads=best_params['n_heads'],
        use_spatial=use_spatial,
        use_chem_attn=use_chem_attn
    ).to(device)

    opt = torch.optim.AdamW(
        abl_model.parameters(),
        lr=best_params['lr'],
        weight_decay=best_params['weight_decay']
    )

    for ep in range(ABLATION_EPOCHS):

        abl_model.train()

        for xb, yrb, ycb, anb in train_loader_a:

            xb = xb.to(device)
            yrb = yrb.to(device)
            ycb = ycb.to(device)
            anb = anb.to(device)

            opt.zero_grad()

            reg_out, cls_out, _, _ = abl_model(xb)

            loss, _, _ = multitask_loss_delta(
                reg_out,
                yrb,
                anb,
                cls_out,
                ycb,
                1.0,
                lambda2
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                abl_model.parameters(),
                max_norm=1.0
            )

            opt.step()

    # ---------- Evaluation ----------
    _, mae, rmse, sm, f1 = evaluate_anchor(test_loader_a, abl_model)

    results_table.append({
        'Model': tag,
        'MAE': mae,
        'RMSE': rmse,
        'SMAPE': sm,
        'MacroF1': f1
    })

    print(
        f'{tag} | MAE {mae:.2f} | '
        f'RMSE {rmse:.2f} | '
        f'SMAPE {sm:.2f} | '
        f'F1 {f1:.3f}'
    )


run_ablation(
    use_spatial=False,
    use_chem_attn=True,
    lambda2=best_params['lambda2'],
    tag='Ablation A (no spatial graph)'
)

run_ablation(
    use_spatial=True,
    use_chem_attn=False,
    lambda2=best_params['lambda2'],
    tag='Ablation B (no chemical attention)'
)

run_ablation(
    use_spatial=True,
    use_chem_attn=True,
    lambda2=0.0,
    tag='Ablation C (unweighted loss)'
)

In [ ]:
# Phase 4: Final Comparison Table + Publication Figures

In [ ]:
# Full comparison: statistical / ML / DL baselines vs proposed model vs ablations
results_df = pd.DataFrame(results_table)
results_df = results_df.round(3)
print(results_df.to_string(index=False))
results_df.to_csv('/kaggle/working/figures/model_comparison.csv', index=False)

# Bar chart of MAE across all models for the report
plt.figure(figsize=(10, 5))
plt.bar(results_df['Model'], results_df['MAE'], color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.ylabel('MAE (AQI units)')
plt.title('Model Comparison — Mean Absolute Error (lower is better)')
plt.tight_layout()
plt.savefig('/kaggle/working/figures/model_comparison_mae.png', dpi=120)
plt.show()


In [ ]:
# ============================================================
# MACHINE UNLEARNING: Remove high-loss / noisy training samples, retrain from scratch
# ============================================================
score_loader = DataLoader(train_ds_a, batch_size=64, shuffle=False)
model.eval()
per_sample_losses = []
with torch.no_grad():
    for xb, yrb, ycb, anb in score_loader:
        xb, yrb, ycb, anb = xb.to(device), yrb.to(device), ycb.to(device), anb.to(device)
        reg_out, cls_out, _, _ = model(xb)
        delta_true = yrb - anb.unsqueeze(1)
        per_example_mse = ((reg_out - delta_true) ** 2).mean(dim=(1, 2))
        per_sample_losses.append(per_example_mse.cpu().numpy())

per_sample_losses = np.concatenate(per_sample_losses)
print('Training sample loss stats -> mean:', per_sample_losses.mean(), '| max:', per_sample_losses.max())

REMOVE_FRACTION = 0.08
threshold = np.percentile(per_sample_losses, 100 * (1 - REMOVE_FRACTION))
keep_mask = per_sample_losses < threshold
print(f'Removing {np.sum(~keep_mask)} / {len(keep_mask)} training samples '
      f'({REMOVE_FRACTION*100:.0f}% highest-loss / most anomalous)')

Xtr_u = Xtr_a[keep_mask]
Yrtr_u = Yrtr_a[keep_mask]
Yctr_u = Yctr_a[keep_mask]
Anctr_u = Anctr[keep_mask]

train_ds_u = AQIDatasetAnchor(Xtr_u, Yrtr_u, Yctr_u, Anctr_u)
train_loader_u = DataLoader(train_ds_u, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

model_unlearned = ST_MPFT(P=P, M=M, T=H, d_model=best_params['d_model'], F_horizon=F_HORIZON,
                           n_classes=N_CLASSES, patch_len=best_params['patch_len'],
                           n_heads=best_params['n_heads'], use_spatial=True, use_chem_attn=True).to(device)
opt_u = torch.optim.Adam(model_unlearned.parameters(), lr=best_params['lr'],
                          weight_decay=best_params['weight_decay'])
sched_u = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_u, mode='min', factor=0.5, patience=5)

UNLEARN_EPOCHS = 100
best_val_loss = float('inf')

# ---------- Early Stopping ----------
patience = 15
counter = 0
best_mae = float('inf')
# ------------------------------------

for epoch in range(UNLEARN_EPOCHS):                          # FIXED: was N_EPOCHS
    model_unlearned.train()
    running_loss = 0.0
    for xb, yrb, ycb, anb in train_loader_u:
        xb, yrb, ycb, anb = xb.to(device), yrb.to(device), ycb.to(device), anb.to(device)
        opt_u.zero_grad()
        reg_out, cls_out, _, _ = model_unlearned(xb)
        loss, _, _ = multitask_loss_delta(reg_out, yrb, anb, cls_out, ycb, LAMBDA1, LAMBDA2)
        loss.backward()
        opt_u.step()
        running_loss += loss.item() * xb.size(0)

    val_loss_u, val_mae_u, val_rmse_u, val_smape_u, val_f1_u = evaluate_anchor(val_loader_a, model_unlearned)
    sched_u.step(val_loss_u)

    if val_loss_u < best_val_loss:                            # FIXED: was best_val_loss_u
        best_val_loss = val_loss_u
        torch.save(model_unlearned.state_dict(), '/kaggle/working/best_st_mpft_unlearned.pt')

    # --- Early stopping check (now actually wired up) ---
    if val_mae_u < best_mae:
        best_mae = val_mae_u
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print(f'Early stopping at Epoch {epoch+1}')
            break

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'[Unlearned] Epoch {epoch+1}/{UNLEARN_EPOCHS} | Val Loss {val_loss_u:.4f} | MAE {val_mae_u:.2f}')

model_unlearned.load_state_dict(torch.load('/kaggle/working/best_st_mpft_unlearned.pt'))
test_loss_u, test_mae_u, test_rmse_u, test_smape_u, test_f1_u = evaluate_anchor(test_loader_a, model_unlearned)
print(f'\nST-MPFT (with unlearning) TEST | MAE {test_mae_u:.2f} | RMSE {test_rmse_u:.2f} '
      f'| SMAPE {test_smape_u:.2f} | Macro F1 {test_f1_u:.3f}')

results_table.append({'Model': 'ST-MPFT (with unlearning)', 'MAE': test_mae_u, 'RMSE': test_rmse_u,
                       'SMAPE': test_smape_u, 'MacroF1': test_f1_u})

In [ ]:
print("P =", P)
print("len(POLLUTANTS) =", len(POLLUTANTS))
print(POLLUTANTS)

In [ ]:
# Learned spatial adjacency + cross-pollutant attention (attention weight matrices for the paper)

model.eval()

sample = next(iter(test_loader))

# Handle both 3-item and 4-item dataloaders
if len(sample) == 3:
    sample_x, _, _ = sample
elif len(sample) == 4:
    sample_x, _, _, _ = sample
else:
    raise ValueError(f"Unexpected batch size returned by test_loader: {len(sample)}")

sample_x = sample_x.to(device)

with torch.no_grad():
    _, _, attn_weights, adapt_adj = model(sample_x)

# ==========================================================
# Plot Learned Spatial Adjacency
# ==========================================================

plt.figure(figsize=(7, 6))

plt.imshow(adapt_adj.detach().cpu().numpy(), cmap='viridis')

plt.colorbar(label='Learned Spatial Weight')

plt.title('Adaptive Spatial Adjacency Matrix (City-to-City)')

plt.xlabel('City Index')
plt.ylabel('City Index')

plt.tight_layout()

plt.savefig(
    '/kaggle/working/figures/spatial_adjacency.png',
    dpi=120,
    bbox_inches='tight'
)

plt.show()

# ==========================================================
# Cross-Pollutant Attention
# ==========================================================

avg_attn = attn_weights[0, -1, 0].detach().cpu().numpy()

num_features = avg_attn.shape[0]

# Use pollutant names only if dimensions match
if len(POLLUTANTS) == num_features:
    labels = POLLUTANTS
else:
    print(f"Warning: Attention matrix is {num_features}x{num_features}, "
          f"but POLLUTANTS has only {len(POLLUTANTS)} labels.")
    labels = [f"Feature {i+1}" for i in range(num_features)]

plt.figure(figsize=(8, 7))

plt.imshow(avg_attn, cmap='magma')

plt.xticks(range(num_features), labels, rotation=45, ha='right')
plt.yticks(range(num_features), labels)

plt.colorbar(label='Attention Weight')

plt.title('Cross-Feature Chemical Attention')

plt.tight_layout()

plt.savefig(
    '/kaggle/working/figures/chemical_attention.png',
    dpi=120,
    bbox_inches='tight'
)

plt.show()

print("City order used for spatial adjacency:")
print(cities)

print("\nAttention Matrix Shape:", avg_attn.shape)
print("Number of model input features:", num_features)

print("\nFigures saved in:")
print("/kaggle/working/figures/")

In [ ]:
# PHASE 5: IMPROVEMENTS